# Basic imports and making use of the TPU if available

(I highly recommend using a TPU and not a CPU because after further developping the model, computations on the TPU that may take 15 seconds take around 100 seconds on CPU)

In [ ]:
# Import torch
import torch
from torch import nn

# Setup device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
!nvidia-smi

Sat Sep  5 16:25:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Preparing the data

In [ ]:
from pathlib import Path

# Setup path to data folder
data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi_100_percent"

# If the image folder doesn't exist, create it
if image_path.is_dir():
    print(f"{image_path} directory exists.")
else:
    print(f"Did not find {image_path} directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)

Did not find data/pizza_steak_sushi_100_percent directory, creating one...


## Manual Step

Here there is a step you have to do manually:

Import the zipped data folder from your computer which you can get by running the code in the data_extraction.ipynb file available in the Sushi-Steak-Pizza-Prediction-model repo. More specifically, put it in the data folder created above.

(I could have also put the file on github, so that I may do everything by code but the file even when zipped was too big to display on github)

In [ ]:
import zipfile

# Unzip pizza, steak, sushi data
with zipfile.ZipFile(data_path / "pizza_steak_sushi_100_percent.zip", "r") as zip_ref:
    print("Unzipping pizza, steak, sushi 100% data...")
    zip_ref.extractall(image_path)

Unzipping pizza, steak, sushi 100% data...


In [ ]:
# 2. Become one with the data
import os
def walk_through_dir(dir_path):
  """Walks through dir_path returning file counts of its contents."""
  for dirpath, dirnames, filenames in os.walk(dir_path):
    print(f"There are {len(dirnames)} directories and {len(filenames)} images in '{dirpath}'.")

In [ ]:
walk_through_dir(image_path)

There are 2 directories and 0 images in 'data/pizza_steak_sushi_100_percent'.
There are 3 directories and 0 images in 'data/pizza_steak_sushi_100_percent/train'.
There are 0 directories and 750 images in 'data/pizza_steak_sushi_100_percent/train/steak'.
There are 0 directories and 750 images in 'data/pizza_steak_sushi_100_percent/train/sushi'.
There are 0 directories and 750 images in 'data/pizza_steak_sushi_100_percent/train/pizza'.
There are 3 directories and 0 images in 'data/pizza_steak_sushi_100_percent/test'.
There are 0 directories and 250 images in 'data/pizza_steak_sushi_100_percent/test/steak'.
There are 0 directories and 250 images in 'data/pizza_steak_sushi_100_percent/test/sushi'.
There are 0 directories and 250 images in 'data/pizza_steak_sushi_100_percent/test/pizza'.


In [ ]:
# Setup train and testing paths

train_dir = image_path / "train"
test_dir = image_path / "test"
train_dir, test_dir

(PosixPath('data/pizza_steak_sushi_100_percent/train'),
 PosixPath('data/pizza_steak_sushi_100_percent/test'))

# Writing the code for training our model

In [ ]:
def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer):

  # Put the model in train mode
  model.train()

  # Setup train loss and train accuracy values
  train_loss, train_acc = 0, 0

  # Loop through data loader and data batches
  for batch, (X, y) in enumerate(dataloader):

    # Send data to target device
    X , y = X.to(device), y.to(device)

    # 1. Forward pass
    y_pred = model(X) # Outputs logits

    # 2. Calculate and accumulate loss
    loss = loss_fn(y_pred, y)
    train_loss += loss.item()

    # 3. Optimizer zero grad
    optimizer.zero_grad()


    # 4. Loss backward
    loss.backward()

    # 5. Optimizer step
    optimizer.step()

    # Calculate and accumualte accuracy metric across all batches
    y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim = 1)
    train_acc += (y_pred_class == y).sum().item()/len(y_pred)

  # Adjust metrics to get average loss and average accuracy per batch
  train_loss = train_loss / len(dataloader)
  train_acc = train_acc / len(dataloader)

  return train_loss, train_acc

In [ ]:
def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module):

  # Put model in eval mode
  model.eval()

  # Setup the test loss and test accuracy values
  test_loss, test_acc = 0, 0

  # Turn on inference context manager
  with torch.inference_mode():

    # Loop through DataLoader batches
    for Batch, (X, y) in enumerate(dataloader):

      # Send data to target device
      X, y = X.to(device), y.to(device)

      # 1. Forward pass
      y_pred = model(X)

      # 2. Calculuate and accumulate loss
      loss = loss_fn(y_pred, y)
      test_loss += loss.item()

      # Calculate and accumulate accuracy
      class_pred = torch.argmax(torch.softmax(y_pred, dim = 1), dim = 1)
      test_acc += (class_pred == y).sum().item()/len(class_pred)

  # Adjust metrics to get average loss and accuracy per batch

  test_loss /= len(dataloader)
  test_acc /= len(dataloader)

  return test_loss, test_acc

In [ ]:
from pathlib import Path

#1. Create model's directory
MODEL_PATH = Path("models")

# Create the models folder where all our best performing models will be saved
MODEL_PATH.mkdir(parents = True, exist_ok = True)



In [ ]:
from tqdm.auto import tqdm
best_acc = 0

def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module = nn.CrossEntropyLoss(),
          best_acc: float = best_acc,
          epochs: int = 5):


  # Create results dictionary
  results = {"train_loss": [],
             "train_acc": [],
             "test_loss": [],
             "test_acc": []}

  # Loop through the training and testing steps for a number of epochs
  for epoch in tqdm(range(epochs)):
    # Train step
    train_loss, train_acc = train_step(model=model,
                                       dataloader=train_dataloader,
                                       loss_fn=loss_fn,
                                       optimizer=optimizer)
    # Test step
    test_loss, test_acc = test_step(model=model,
                                    dataloader=test_dataloader,
                                    loss_fn=loss_fn)

    # Save model if its performance is good

    minimum_acc = 0.90

    if test_acc > minimum_acc and test_acc > best_acc:
      best_acc = test_acc

      # Save the model's with their test accuracy in their file name
      acc = test_acc * 100
      acc_int = int(acc)

      acc_float = int((acc-acc_int) * 100)

      MODEL_NAME = f"model_{acc_int}-{acc_float}.pth"
      MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

      print(f"New high: {best_acc}, Saving model to : {MODEL_SAVE_PATH}")

      torch.save(obj=model.state_dict(),
            f = MODEL_SAVE_PATH)

    # Print out what's happening
    print(f"Epoch: {epoch+1} | "
          f"train_loss: {train_loss:.4f} | "
          f"train_acc: {train_acc:.4f} | "
          f"test_loss: {test_loss:.4f} | "
          f"test_acc: {test_acc:.4f}"
    )

    # Update the results dictionary
    results["train_loss"].append(train_loss)
    results["train_acc"].append(train_acc)
    results["test_loss"].append(test_loss)
    results["test_acc"].append(test_acc)

  # Return the results dictionary
  return results

# Start our experimentation

## Model 00: 10 Hidden units, basic data augmentation

In [ ]:
from torch import nn
class TinyVGG_0_64(nn.Module):
  """
  Model architechture copying TinyVGG from CNN Explainer
  """
  def __init__(self,
               input_shape : int,
               hidden_units : int,
               output_shape : int) -> None:

    super().__init__()

    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels = input_shape,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 0),
        nn.ReLU(),
        nn.Conv2d(in_channels = hidden_units,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 0),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2,
                     stride = 2)
    )

    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(
            in_channels = hidden_units,
            out_channels = hidden_units,
            kernel_size = 3,
            stride = 1,
            padding = 0
        ),
        nn.ReLU(),
        nn.Conv2d(in_channels = hidden_units,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 0),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2,
                     stride = 2)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features = hidden_units * 169,
                  out_features = output_shape)
    )

  def forward(self, x):
    x = self.conv_block_1(x)
    x = self.conv_block_2(x)
    x = self.classifier(x)
    return x

In [ ]:
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import os

augmented_transform_1_64 = transforms.Compose([
    transforms.Resize(size = (64, 64)),
    transforms.TrivialAugmentWide(num_magnitude_bins = 5),
    transforms.ToTensor()
])

simple_transform_64 = transforms.Compose([
    transforms.Resize(size = (64, 64)),
    transforms.ToTensor()
])

train_data_1 = datasets.ImageFolder(
    root = train_dir,
    transform = augmented_transform_1_64,
)

test_data_1 = datasets.ImageFolder(
    root = test_dir,
    transform = simple_transform_64
)

BATCH_SIZE = 16

train_dataloader_1 = DataLoader(
    dataset = train_data_1,
    batch_size = BATCH_SIZE,
    shuffle = True,
    num_workers = os.cpu_count()
)

test_dataloader_1 = DataLoader(
    dataset = test_data_1,
    batch_size = BATCH_SIZE,
    shuffle = False,
    num_workers = os.cpu_count()
)

In [ ]:
# Set the random seeds for reproduceability
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model_00 = TinyVGG_0_64(
    input_shape = 3,
    hidden_units = 10,
    output_shape = 3
).to(device)


In [ ]:
optimizer_00 = torch.optim.Adam(params = model_00.parameters(),
                             lr = 0.001)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

train(model = model_00,
      train_dataloader = train_dataloader_1,
      test_dataloader = test_dataloader_1,
      optimizer = optimizer_00,
      loss_fn = loss_fn,
      epochs = 50)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 1.0704 | train_acc: 0.4012 | test_loss: 0.9425 | test_acc: 0.5426
Epoch: 2 | train_loss: 0.9932 | train_acc: 0.4934 | test_loss: 0.9045 | test_acc: 0.5414
Epoch: 3 | train_loss: 0.9715 | train_acc: 0.5049 | test_loss: 0.8952 | test_acc: 0.5963
Epoch: 4 | train_loss: 0.9554 | train_acc: 0.5221 | test_loss: 0.8552 | test_acc: 0.6168
Epoch: 5 | train_loss: 0.9683 | train_acc: 0.5315 | test_loss: 0.9005 | test_acc: 0.5843
Epoch: 6 | train_loss: 0.9372 | train_acc: 0.5590 | test_loss: 0.8581 | test_acc: 0.6109
Epoch: 7 | train_loss: 0.9274 | train_acc: 0.5576 | test_loss: 0.8403 | test_acc: 0.6165
Epoch: 8 | train_loss: 0.9118 | train_acc: 0.5654 | test_loss: 0.8593 | test_acc: 0.6123
Epoch: 9 | train_loss: 0.9139 | train_acc: 0.5609 | test_loss: 0.8150 | test_acc: 0.6337
Epoch: 10 | train_loss: 0.8817 | train_acc: 0.5777 | test_loss: 0.8070 | test_acc: 0.6222
Epoch: 11 | train_loss: 0.8862 | train_acc: 0.5886 | test_loss: 0.8019 | test_acc: 0.6351
Epoch: 12 | train_l

{'train_loss': [1.0703969648543825,
  0.9931643884232704,
  0.9715029626873368,
  0.9553598926422444,
  0.9682646871458555,
  0.9372157818036722,
  0.9274324825469483,
  0.9117560919294966,
  0.9139181037321158,
  0.8816853617945462,
  0.8862270070305953,
  0.8744412910008261,
  0.8879730595764539,
  0.8644001585371951,
  0.8775010899449072,
  0.860695684632511,
  0.8428689309045778,
  0.8485321349708747,
  0.8439602501003455,
  0.8318040349804763,
  0.8313556636901612,
  0.8105294330322996,
  0.8013733173093052,
  0.8240641638742271,
  0.8023792120581823,
  0.7886299469792251,
  0.8098746326798243,
  0.7808553238287039,
  0.7781863675472585,
  0.7691183946234115,
  0.7872700439699998,
  0.7862946640515158,
  0.760977845242683,
  0.7385938881982302,
  0.7549094407693714,
  0.7430677688713615,
  0.7378669076777519,
  0.7186710208865768,
  0.7427095301607822,
  0.726498248517936,
  0.7365398819142199,
  0.7411405355372327,
  0.7105011092432847,
  0.7094518464085058,
  0.7199660882036737,

Save the model's performance in the last few epochs


## Model 100: 10 hidden units, num_magnitude_bins = 31

#### Perhaps more data augmentation is required?

In [ ]:
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import os

augmented_transform_2_64 = transforms.Compose([
    transforms.Resize(size = (64, 64)),
    transforms.TrivialAugmentWide(num_magnitude_bins = 31),
    transforms.ToTensor()
])

simple_transform_64 = transforms.Compose([
    transforms.Resize(size = (64, 64)),
    transforms.ToTensor()
])

train_data_2 = datasets.ImageFolder(
    root = train_dir,
    transform = augmented_transform_2_64,
)

test_data_1 = datasets.ImageFolder(
    root = test_dir,
    transform = simple_transform_64
)

BATCH_SIZE = 16

train_dataloader_2 = DataLoader(
    dataset = train_data_1,
    batch_size = BATCH_SIZE,
    shuffle = True,
    num_workers = os.cpu_count()
)

test_dataloader_1 = DataLoader(
    dataset = test_data_1,
    batch_size = BATCH_SIZE,
    shuffle = False,
    num_workers = os.cpu_count()
)

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model_100 = TinyVGG_0_64(
    input_shape = 3,
    hidden_units = 10,
    output_shape = 3
).to(device)


In [ ]:
optimizer_100 = torch.optim.Adam(params = model_100.parameters(),
                             lr = 0.001)
loss_fn = nn.CrossEntropyLoss()


In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

train(model = model_100,
      train_dataloader = train_dataloader_1,
      test_dataloader = test_dataloader_1,
      optimizer = optimizer_100,
      loss_fn = loss_fn,
      epochs = 50)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 1.0982 | train_acc: 0.3430 | test_loss: 1.0971 | test_acc: 0.3351
Epoch: 2 | train_loss: 1.0765 | train_acc: 0.3969 | test_loss: 0.9587 | test_acc: 0.5570
Epoch: 3 | train_loss: 0.9904 | train_acc: 0.5103 | test_loss: 0.9465 | test_acc: 0.5256
Epoch: 4 | train_loss: 0.9542 | train_acc: 0.5417 | test_loss: 0.8959 | test_acc: 0.5881
Epoch: 5 | train_loss: 0.9331 | train_acc: 0.5564 | test_loss: 0.8552 | test_acc: 0.6237
Epoch: 6 | train_loss: 0.9349 | train_acc: 0.5451 | test_loss: 0.8437 | test_acc: 0.6394
Epoch: 7 | train_loss: 0.9213 | train_acc: 0.5571 | test_loss: 0.8238 | test_acc: 0.6322
Epoch: 8 | train_loss: 0.9146 | train_acc: 0.5619 | test_loss: 0.7768 | test_acc: 0.6436
Epoch: 9 | train_loss: 0.9057 | train_acc: 0.5723 | test_loss: 0.8429 | test_acc: 0.6136
Epoch: 10 | train_loss: 0.8980 | train_acc: 0.5743 | test_loss: 0.7883 | test_acc: 0.6396
Epoch: 11 | train_loss: 0.8874 | train_acc: 0.5943 | test_loss: 0.7525 | test_acc: 0.6704
Epoch: 12 | train_l

{'train_loss': [1.098219354947408,
  1.0764574467713106,
  0.990421325179702,
  0.954208252700508,
  0.9330691027303114,
  0.9348796187563145,
  0.921273318165583,
  0.9146181860714094,
  0.9057200210736999,
  0.8979748491699814,
  0.8874137667899437,
  0.8735411877327777,
  0.8698041840647974,
  0.8652080792061826,
  0.8810757504287341,
  0.8408433207383392,
  0.8716671952964566,
  0.8423737980795245,
  0.8270137715846935,
  0.8291435977245899,
  0.8179611316386689,
  0.8162570728900584,
  0.8116835482577061,
  0.8117107835644526,
  0.7860192603676032,
  0.7852617544485322,
  0.7731753591950058,
  0.7785960304821636,
  0.7716692946058639,
  0.7724196300016227,
  0.7549234290494986,
  0.7459978290060734,
  0.7462144058646886,
  0.7499317182294021,
  0.7302517578111473,
  0.7161841236107739,
  0.72772287011992,
  0.7195301244022153,
  0.7090800406662285,
  0.6909017233138389,
  0.6991460399424776,
  0.7057215560835304,
  0.6798349094729051,
  0.6694441765335435,
  0.67148889886572,
  0.


Epoch: 45 | train_loss: 0.6715 | train_acc: 0.7205 | test_loss: 0.6569 | test_acc: 0.7213

Epoch: 46 | train_loss: 0.6763 | train_acc: 0.7152 | test_loss: 0.6655 | test_acc: 0.7209

Epoch: 47 | train_loss: 0.6951 | train_acc: 0.6916 | test_loss: 0.6959 | test_acc: 0.6970

Epoch: 48 | train_loss: 0.6738 | train_acc: 0.7129 | test_loss: 0.6293 | test_acc: 0.7405

Epoch: 49 | train_loss: 0.6442 | train_acc: 0.7248 | test_loss: 0.6386 | test_acc: 0.7572

Epoch: 50 | train_loss: 0.6531 | train_acc: 0.7336 | test_loss: 0.6479 | test_acc: 0.7342

## Model 200: 32 hidden units and num_magnitude_bins = 31

#### Upgrade the model's computational power

In [ ]:
augmented_transform_2_64 = transforms.Compose([
    transforms.Resize(size = (64, 64)),
    transforms.TrivialAugmentWide(num_magnitude_bins = 31),
    transforms.ToTensor()
])

simple_transform_2_64 = transforms.Compose([
    transforms.Resize(size = (64, 64)),
    transforms.ToTensor()
])

train_data_2_64 = datasets.ImageFolder(
    root = train_dir,
    transform = augmented_transform_2_64,
)

test_data_2_64 = datasets.ImageFolder(
    root = test_dir,
    transform = simple_transform_2_64
)

BATCH_SIZE = 16

train_dataloader_2_64 = DataLoader(
    dataset = train_data_2_64,
    batch_size = BATCH_SIZE,
    shuffle = True,
    num_workers = os.cpu_count()
)

test_dataloader_2_64 = DataLoader(
    dataset = test_data_2_64,
    batch_size = BATCH_SIZE,
    shuffle = False,
    num_workers = os.cpu_count()
)

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model_200 = TinyVGG_0_64(input_shape = 3,
                    hidden_units= 32,
                    output_shape = 3).to(device)

In [ ]:
optimizer_200 = torch.optim.Adam(params = model_200.parameters(),
                             lr = 0.001)
loss_fn = nn.CrossEntropyLoss()


In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

train(model = model_200,
      train_dataloader = train_dataloader_2_64,
      test_dataloader = test_dataloader_2_64,
      optimizer = optimizer_200,
      loss_fn = loss_fn,
      epochs = 50)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 1.0293 | train_acc: 0.4570 | test_loss: 0.9013 | test_acc: 0.5828
Epoch: 2 | train_loss: 0.9627 | train_acc: 0.5170 | test_loss: 0.9384 | test_acc: 0.5294
Epoch: 3 | train_loss: 0.9291 | train_acc: 0.5590 | test_loss: 0.9192 | test_acc: 0.5521
Epoch: 4 | train_loss: 0.9326 | train_acc: 0.5554 | test_loss: 0.7943 | test_acc: 0.6413
Epoch: 5 | train_loss: 0.8892 | train_acc: 0.5882 | test_loss: 0.8416 | test_acc: 0.5999
Epoch: 6 | train_loss: 0.8718 | train_acc: 0.5986 | test_loss: 0.7623 | test_acc: 0.6714
Epoch: 7 | train_loss: 0.8623 | train_acc: 0.6030 | test_loss: 0.7640 | test_acc: 0.6480
Epoch: 8 | train_loss: 0.8613 | train_acc: 0.5951 | test_loss: 0.7814 | test_acc: 0.6611
Epoch: 9 | train_loss: 0.8333 | train_acc: 0.6152 | test_loss: 0.7227 | test_acc: 0.6780
Epoch: 10 | train_loss: 0.8032 | train_acc: 0.6511 | test_loss: 0.6764 | test_acc: 0.6940
Epoch: 11 | train_loss: 0.7847 | train_acc: 0.6513 | test_loss: 0.6779 | test_acc: 0.6970
Epoch: 12 | train_l

{'train_loss': [1.029303435315477,
  0.9627144603019065,
  0.9290926346542142,
  0.9325662149605176,
  0.8892006696538722,
  0.87183803954023,
  0.8622554181315375,
  0.8613456876565379,
  0.8333150604937939,
  0.8032426438855786,
  0.7846935484426242,
  0.7872679827483833,
  0.7535206726256837,
  0.7464808164336157,
  0.7252078343790473,
  0.7252341389444703,
  0.7034520718222814,
  0.6824938745786112,
  0.7049394104074924,
  0.6454518221794291,
  0.64169464910284,
  0.626594568186618,
  0.6371425498039165,
  0.6147122607163503,
  0.6133977796169038,
  0.6316525669808083,
  0.5904531312961105,
  0.5517285561307947,
  0.5641798828299164,
  0.5688791321524491,
  0.5286271816238444,
  0.5634686041174205,
  0.5693631142589217,
  0.5414804518222809,
  0.5290160390502172,
  0.5025860093375469,
  0.5082833729948558,
  0.48890051002620805,
  0.5283192442664018,
  0.5033082902854216,
  0.48373774201311964,
  0.49471674701000784,
  0.4641116584869141,
  0.47292277047820125,
  0.4595104258322546

Epoch: 45 | train_loss: 0.4595 | train_acc: 0.8223 | test_loss: 0.6078 | test_acc: 0.7654

Epoch: 46 | train_loss: 0.4784 | train_acc: 0.8167 | test_loss: 0.5755 | test_acc: 0.7720

Epoch: 47 | train_loss: 0.4386 | train_acc: 0.8315 | test_loss: 0.6171 | test_acc: 0.7815

Epoch: 48 | train_loss: 0.4403 | train_acc: 0.8300 | test_loss: 0.5961 | test_acc: 0.7924

Epoch: 49 | train_loss: 0.4701 | train_acc: 0.8239 | test_loss: 0.6181 | test_acc: 0.7399

Epoch: 50 | train_loss: 0.4461 | train_acc: 0.8331 | test_loss: 0.5601 | test_acc: 0.7882

 Lower the learning rate and train the model again to further increase his performance

In [ ]:
optimizer_200 = torch.optim.Adam(params = model_200.parameters(),
                             lr = 0.0005)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
train(model = model_200,
      train_dataloader = train_dataloader_2_64,
      test_dataloader = test_dataloader_2_64,
      optimizer = optimizer_200,
      loss_fn = loss_fn,
      epochs = 50)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 0.4023 | train_acc: 0.8461 | test_loss: 0.5995 | test_acc: 0.7870
Epoch: 2 | train_loss: 0.4067 | train_acc: 0.8485 | test_loss: 0.5738 | test_acc: 0.7817
Epoch: 3 | train_loss: 0.4221 | train_acc: 0.8353 | test_loss: 0.5997 | test_acc: 0.7764
Epoch: 4 | train_loss: 0.4306 | train_acc: 0.8421 | test_loss: 0.5365 | test_acc: 0.7804
Epoch: 5 | train_loss: 0.4057 | train_acc: 0.8477 | test_loss: 0.5202 | test_acc: 0.8030
Epoch: 6 | train_loss: 0.4081 | train_acc: 0.8468 | test_loss: 0.5834 | test_acc: 0.7846
Epoch: 7 | train_loss: 0.3696 | train_acc: 0.8623 | test_loss: 0.6205 | test_acc: 0.7692
Epoch: 8 | train_loss: 0.3852 | train_acc: 0.8582 | test_loss: 0.5262 | test_acc: 0.8017
Epoch: 9 | train_loss: 0.4020 | train_acc: 0.8490 | test_loss: 0.5767 | test_acc: 0.7840
Epoch: 10 | train_loss: 0.3958 | train_acc: 0.8421 | test_loss: 0.5528 | test_acc: 0.7964
Epoch: 11 | train_loss: 0.3964 | train_acc: 0.8561 | test_loss: 0.5122 | test_acc: 0.8176
Epoch: 12 | train_l

{'train_loss': [0.4022914732390262,
  0.40670119653990927,
  0.42208923589675984,
  0.43063010597694007,
  0.40566957002201826,
  0.40813309102193684,
  0.3695771211216636,
  0.38519924725835203,
  0.4020215893256749,
  0.39581412409848354,
  0.39643421869540046,
  0.37459997716524923,
  0.3807786039955227,
  0.40871844588653417,
  0.3839666352521443,
  0.3715740886140377,
  0.3750660009946383,
  0.3890747113536436,
  0.3718568934986355,
  0.35892503026952133,
  0.3847081052496078,
  0.3622697229820786,
  0.3907167102533875,
  0.3308450219267649,
  0.3879360597818456,
  0.3578521539556219,
  0.35634357022478225,
  0.3526127753211251,
  0.33774746428673147,
  0.3489140477163572,
  0.33732001420031205,
  0.3735468217984159,
  0.34759418069259496,
  0.31571588115700594,
  0.36056352100262407,
  0.32992138824564343,
  0.3512091119080148,
  0.3321074595318196,
  0.32999523855904317,
  0.36637834755452814,
  0.32801075295564974,
  0.34545443117195834,
  0.34359163307446117,
  0.3430563364679

Epoch: 45 | train_loss: 0.3264 | train_acc: 0.8759 | test_loss: 0.5760 | test_acc: 0.8055

Epoch: 46 | train_loss: 0.3135 | train_acc: 0.8852 | test_loss: 0.5273 | test_acc: 0.8136

Epoch: 47 | train_loss: 0.3210 | train_acc: 0.8821 | test_loss: 0.5449 | test_acc: 0.8015

Epoch: 48 | train_loss: 0.3331 | train_acc: 0.8756 | test_loss: 0.5407 | test_acc: 0.7895

Epoch: 49 | train_loss: 0.3169 | train_acc: 0.8811 | test_loss: 0.5790 | test_acc: 0.8068

Epoch: 50 | train_loss: 0.3297 | train_acc: 0.8734 | test_loss: 0.5533 | test_acc: 0.8136


## Model 300: 32 hidden units, num_magnitude_bins = 31 and 96 pixel images

### Make the image quality better so that the model has more information to work on

In [ ]:
from torch import nn
class TinyVGG_0_96(nn.Module):
  """
  Model architechture copying TinyVGG from CNN Explainer
  """
  def __init__(self,
               input_shape : int,
               hidden_units : int,
               output_shape : int) -> None:

    super().__init__()

    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels = input_shape,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 0),
        nn.ReLU(),
        nn.Conv2d(in_channels = hidden_units,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 0),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2,
                     stride = 2)
    )

    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(
            in_channels = hidden_units,
            out_channels = hidden_units,
            kernel_size = 3,
            stride = 1,
            padding = 0
        ),
        nn.ReLU(),
        nn.Conv2d(in_channels = hidden_units,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 0),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2,
                     stride = 2)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features = 14112,
                  out_features = output_shape)
    )

  def forward(self, x):
    x = self.conv_block_1(x)
    x = self.conv_block_2(x)
    x = self.classifier(x)
    return x

In [ ]:
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader

augmented_transform_2_96 = transforms.Compose([
    transforms.Resize(size = (96, 96)),
    transforms.TrivialAugmentWide(num_magnitude_bins = 31),
    transforms.ToTensor()
])

simple_transform_96 = transforms.Compose([
    transforms.Resize(size = (96, 96)),
    transforms.ToTensor()
])

train_data_2_96 = datasets.ImageFolder(
    root = train_dir,
    transform = augmented_transform_2_96,
)

test_data_2_96 = datasets.ImageFolder(
    root = test_dir,
    transform = simple_transform_96
)

BATCH_SIZE = 16

train_dataloader_2_96 = DataLoader(
    dataset = train_data_2_96,
    batch_size = BATCH_SIZE,
    shuffle = True,
    num_workers = os.cpu_count()
)

test_dataloader_2_96 = DataLoader(
    dataset = test_data_2_96,
    batch_size = BATCH_SIZE,
    shuffle = False,
    num_workers = os.cpu_count()
)

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model_300 = TinyVGG_0_96(input_shape = 3,
                    hidden_units= 32,
                    output_shape = 3).to(device)

In [ ]:
optimizer_300 = torch.optim.Adam(params = model_300.parameters(),
                             lr = 0.001)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

train(model = model_300,
      train_dataloader = train_dataloader_2_96,
      test_dataloader = test_dataloader_2_96,
      optimizer = optimizer_300,
      loss_fn = loss_fn,
      epochs = 50)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 1.0605 | train_acc: 0.4214 | test_loss: 0.9741 | test_acc: 0.5452
Epoch: 2 | train_loss: 0.9893 | train_acc: 0.4915 | test_loss: 0.8920 | test_acc: 0.5748
Epoch: 3 | train_loss: 0.9564 | train_acc: 0.5362 | test_loss: 0.8731 | test_acc: 0.5830
Epoch: 4 | train_loss: 0.9444 | train_acc: 0.5538 | test_loss: 0.8469 | test_acc: 0.6341
Epoch: 5 | train_loss: 0.9008 | train_acc: 0.5840 | test_loss: 0.8204 | test_acc: 0.6356
Epoch: 6 | train_loss: 0.8958 | train_acc: 0.5790 | test_loss: 0.8544 | test_acc: 0.6071
Epoch: 7 | train_loss: 0.8719 | train_acc: 0.5956 | test_loss: 0.7772 | test_acc: 0.6556
Epoch: 8 | train_loss: 0.8511 | train_acc: 0.6098 | test_loss: 0.7663 | test_acc: 0.6565
Epoch: 9 | train_loss: 0.8499 | train_acc: 0.6100 | test_loss: 0.7454 | test_acc: 0.6594
Epoch: 10 | train_loss: 0.8207 | train_acc: 0.6462 | test_loss: 0.7400 | test_acc: 0.6619
Epoch: 11 | train_loss: 0.7811 | train_acc: 0.6524 | test_loss: 0.7093 | test_acc: 0.7092
Epoch: 12 | train_l

{'train_loss': [1.0604742465289771,
  0.9893122078679132,
  0.9563510033255773,
  0.9443579810730954,
  0.9007567830965029,
  0.8958067412072039,
  0.8718558173653082,
  0.851105513724875,
  0.8498958230864072,
  0.8207365447747792,
  0.7810878768457589,
  0.7848327612200527,
  0.7600126581411835,
  0.7598362051426096,
  0.7354574926356052,
  0.7345736658742242,
  0.705302415058968,
  0.6777287432065247,
  0.6910259949822798,
  0.6699028796126657,
  0.6694170602247225,
  0.6554666949924848,
  0.6268339767946419,
  0.61485903187001,
  0.6157240099306648,
  0.5949382898232616,
  0.5964735262360134,
  0.5768601687241953,
  0.5524548519376322,
  0.5634892834416518,
  0.5707679402532307,
  0.578549345334371,
  0.5626148180970063,
  0.5450178260076131,
  0.5630639901397921,
  0.5107427070326839,
  0.5187433252626277,
  0.5072252280111854,
  0.49823339019261353,
  0.5194776848698339,
  0.5086963959830872,
  0.5053916806024863,
  0.47321029176526036,
  0.48353985427541935,
  0.4962749194802967

Epoch: 45 | train_loss: 0.4963 | train_acc: 0.8094 | test_loss: 0.6872 | test_acc: 0.7276
Epoch: 46 | train_loss: 0.4675 | train_acc: 0.8257 | test_loss: 0.6783 | test_acc: 0.7375
Epoch: 47 | train_loss: 0.4916 | train_acc: 0.8125 | test_loss: 0.6794 | test_acc: 0.7238
Epoch: 48 | train_loss: 0.4901 | train_acc: 0.8045 | test_loss: 0.6647 | test_acc: 0.7375
Epoch: 49 | train_loss: 0.4370 | train_acc: 0.8361 | test_loss: 0.6976 | test_acc: 0.7266
Epoch: 50 | train_loss: 0.4909 | train_acc: 0.8158 | test_loss: 0.7449 | test_acc: 0.6997

In [ ]:
optimizer_300 = torch.optim.Adam(params = model_300.parameters(),
                             lr = 0.0005)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
train(model = model_300,
      train_dataloader = train_dataloader_2_96,
      test_dataloader = test_dataloader_2_96,
      optimizer = optimizer_300,
      loss_fn = loss_fn,
      epochs = 50)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 0.4267 | train_acc: 0.8402 | test_loss: 0.6893 | test_acc: 0.7397
Epoch: 2 | train_loss: 0.4113 | train_acc: 0.8430 | test_loss: 0.6831 | test_acc: 0.7397
Epoch: 3 | train_loss: 0.4192 | train_acc: 0.8361 | test_loss: 0.6588 | test_acc: 0.7318
Epoch: 4 | train_loss: 0.4166 | train_acc: 0.8402 | test_loss: 0.6308 | test_acc: 0.7585
Epoch: 5 | train_loss: 0.4228 | train_acc: 0.8379 | test_loss: 0.6326 | test_acc: 0.7519
Epoch: 6 | train_loss: 0.4165 | train_acc: 0.8417 | test_loss: 0.6642 | test_acc: 0.7437
Epoch: 7 | train_loss: 0.4290 | train_acc: 0.8328 | test_loss: 0.6178 | test_acc: 0.7466
Epoch: 8 | train_loss: 0.3962 | train_acc: 0.8519 | test_loss: 0.6428 | test_acc: 0.7601
Epoch: 9 | train_loss: 0.3894 | train_acc: 0.8582 | test_loss: 0.6337 | test_acc: 0.7519
Epoch: 10 | train_loss: 0.3983 | train_acc: 0.8523 | test_loss: 0.6289 | test_acc: 0.7599
Epoch: 11 | train_loss: 0.4095 | train_acc: 0.8520 | test_loss: 0.6300 | test_acc: 0.7637
Epoch: 12 | train_l

{'train_loss': [0.4267264755693733,
  0.41131987861284974,
  0.4192477969307426,
  0.4165862596838187,
  0.42276589984589436,
  0.4165047551934601,
  0.42897192859057837,
  0.39624210811675864,
  0.3893519006933726,
  0.39826279675178494,
  0.40952542429486066,
  0.4025196140328198,
  0.40334316586137664,
  0.39586525949391915,
  0.3931508999027259,
  0.40079458892768155,
  0.3695396357711325,
  0.38286940124652064,
  0.4002115835851811,
  0.38393488436514606,
  0.37590354268855236,
  0.3590014192757877,
  0.38045979560689724,
  0.3744549218116077,
  0.38049830033952464,
  0.3751886471683252,
  0.36752868966853364,
  0.3610087194552658,
  0.3645992650524944,
  0.3834109087573721,
  0.37055126625172635,
  0.35863388366733034,
  0.3630222331019158,
  0.3623793189090194,
  0.3414668991950387,
  0.3542536312965214,
  0.34815247322544984,
  0.35427294579063745,
  0.382333738214158,
  0.3494506561005792,
  0.3360567851163817,
  0.38604654691426465,
  0.3318013417012725,
  0.3376813324577842,

Epoch: 45 | train_loss: 0.3490 | train_acc: 0.8725 | test_loss: 0.5847 | test_acc: 0.7743

Epoch: 46 | train_loss: 0.3385 | train_acc: 0.8684 | test_loss: 0.5973 | test_acc: 0.7804

Epoch: 47 | train_loss: 0.3583 | train_acc: 0.8667 | test_loss: 0.5871 | test_acc: 0.7745

Epoch: 48 | train_loss: 0.3626 | train_acc: 0.8685 | test_loss: 0.5643 | test_acc: 0.8000

Epoch: 49 | train_loss: 0.3540 | train_acc: 0.8687 | test_loss: 0.5961 | test_acc: 0.7703

Epoch: 50 | train_loss: 0.3490 | train_acc: 0.8731 | test_loss: 0.5997 | test_acc: 0.7770


## Model 400: 32 hidden units, num_magnitude_bins = 31 and imgs with 128 pxs

In [ ]:
from torch import nn
class TinyVGG_0_128(nn.Module):
  """
  Model architechture copying TinyVGG from CNN Explainer
  """
  def __init__(self,
               input_shape : int,
               hidden_units : int,
               output_shape : int) -> None:

    super().__init__()

    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels = input_shape,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 0),
        nn.ReLU(),
        nn.Conv2d(in_channels = hidden_units,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 0),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2,
                     stride = 2)
    )

    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(
            in_channels = hidden_units,
            out_channels = hidden_units,
            kernel_size = 3,
            stride = 1,
            padding = 0
        ),
        nn.ReLU(),
        nn.Conv2d(in_channels = hidden_units,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 0),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2,
                     stride = 2)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features = 26912,
                  out_features = output_shape)
    )

  def forward(self, x):
    x = self.conv_block_1(x)
    x = self.conv_block_2(x)
    x = self.classifier(x)
    return x

In [ ]:
augmented_transform_2_128 = transforms.Compose([
    transforms.Resize(size = (128, 128)),
    transforms.TrivialAugmentWide(num_magnitude_bins = 31),
    transforms.ToTensor()
])

simple_transform_2_128 = transforms.Compose([
    transforms.Resize(size = (128, 128)),
    transforms.ToTensor()
])

train_data_2_128 = datasets.ImageFolder(
    root = train_dir,
    transform = augmented_transform_2_128,
)

test_data_2_128 = datasets.ImageFolder(
    root = test_dir,
    transform = simple_transform_2_128
)

BATCH_SIZE = 16

train_dataloader_2_128 = DataLoader(
    dataset = train_data_2_128,
    batch_size = BATCH_SIZE,
    shuffle = True,
    num_workers = os.cpu_count()
)

test_dataloader_2_128 = DataLoader(
    dataset = test_data_2_128,
    batch_size = BATCH_SIZE,
    shuffle = False,
    num_workers = os.cpu_count()
)

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model_400 = TinyVGG_0_128(input_shape = 3,
                    hidden_units= 32,
                    output_shape = 3).to(device)

In [ ]:
optimizer_400 = torch.optim.Adam(params = model_400.parameters(),
                             lr = 0.001)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

train(model = model_400,
      train_dataloader = train_dataloader_2_128,
      test_dataloader = test_dataloader_2_128,
      optimizer = optimizer_400,
      loss_fn = loss_fn,
      epochs = 50)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 1.0102 | train_acc: 0.4784 | test_loss: 0.9253 | test_acc: 0.5564
Epoch: 2 | train_loss: 0.9394 | train_acc: 0.5479 | test_loss: 0.8548 | test_acc: 0.6151
Epoch: 3 | train_loss: 0.9215 | train_acc: 0.5693 | test_loss: 0.8205 | test_acc: 0.6379
Epoch: 4 | train_loss: 0.9237 | train_acc: 0.5563 | test_loss: 0.8211 | test_acc: 0.6297
Epoch: 5 | train_loss: 0.9006 | train_acc: 0.5767 | test_loss: 0.8605 | test_acc: 0.5988
Epoch: 6 | train_loss: 0.8896 | train_acc: 0.5895 | test_loss: 0.8208 | test_acc: 0.5919
Epoch: 7 | train_loss: 0.8855 | train_acc: 0.5968 | test_loss: 0.7592 | test_acc: 0.6799
Epoch: 8 | train_loss: 0.8662 | train_acc: 0.6040 | test_loss: 0.7548 | test_acc: 0.6696
Epoch: 9 | train_loss: 0.8510 | train_acc: 0.6060 | test_loss: 0.7414 | test_acc: 0.6771
Epoch: 10 | train_loss: 0.8051 | train_acc: 0.6395 | test_loss: 0.7039 | test_acc: 0.7046
Epoch: 11 | train_loss: 0.7862 | train_acc: 0.6512 | test_loss: 0.6773 | test_acc: 0.7323
Epoch: 12 | train_l

{'train_loss': [1.010242110871254,
  0.9393828020873645,
  0.9215434358475056,
  0.9236674520140844,
  0.900586059541567,
  0.8896046044133233,
  0.8854731029652535,
  0.8661652059419781,
  0.8509985834148759,
  0.8050755651284617,
  0.7862029155940874,
  0.7642628973257457,
  0.7546967399035786,
  0.7226881124871842,
  0.7108271779320764,
  0.6860730199949115,
  0.672200120721303,
  0.6594753358381015,
  0.6165478565591447,
  0.5901357617149962,
  0.5686318557313148,
  0.541598009302261,
  0.5203108043535382,
  0.48809145829567674,
  0.5108551718030415,
  0.5118352812021336,
  0.48566429232451935,
  0.47581152644351865,
  0.4604490646128113,
  0.44680709154047865,
  0.4384186342886999,
  0.43835723843980345,
  0.42509850014186074,
  0.4177833669785912,
  0.4342755322337996,
  0.41770033876523904,
  0.457793967573778,
  0.4022213822455271,
  0.40708459638957434,
  0.39902550711276685,
  0.381972148486063,
  0.3608209288659248,
  0.39460826897663426,
  0.38057889699513187,
  0.355152754

Epoch: 45 | train_loss: 0.3552 | train_acc: 0.8663 | test_loss: 0.5941 | test_acc: 0.7857

Epoch: 46 | train_loss: 0.3852 | train_acc: 0.8576 | test_loss: 0.6350 | test_acc: 0.7696

Epoch: 47 | train_loss: 0.3764 | train_acc: 0.8550 | test_loss: 0.5656 | test_acc: 0.7869

Epoch: 48 | train_loss: 0.3507 | train_acc: 0.8730 | test_loss: 0.5899 | test_acc: 0.7775

Epoch: 49 | train_loss: 0.3618 | train_acc: 0.8656 | test_loss: 0.7757 | test_acc: 0.7403

Epoch: 50 | train_loss: 0.4018 | train_acc: 0.8595 | test_loss: 0.6099 | test_acc: 0.7698


In [ ]:
optimizer_400 = torch.optim.Adam(params = model_400.parameters(),
                             lr = 0.0005)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
train(model = model_400,
      train_dataloader = train_dataloader_2_128,
      test_dataloader = test_dataloader_2_128,
      optimizer = optimizer_400,
      loss_fn = loss_fn,
      epochs = 50)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 0.3158 | train_acc: 0.8824 | test_loss: 0.5748 | test_acc: 0.7937
Epoch: 2 | train_loss: 0.3260 | train_acc: 0.8909 | test_loss: 0.5493 | test_acc: 0.8148
Epoch: 3 | train_loss: 0.3234 | train_acc: 0.8824 | test_loss: 0.5540 | test_acc: 0.8057
Epoch: 4 | train_loss: 0.3018 | train_acc: 0.8918 | test_loss: 0.5564 | test_acc: 0.8123
Epoch: 5 | train_loss: 0.3358 | train_acc: 0.8792 | test_loss: 0.5669 | test_acc: 0.8110
Epoch: 6 | train_loss: 0.2832 | train_acc: 0.8953 | test_loss: 0.5705 | test_acc: 0.8070
Epoch: 7 | train_loss: 0.2991 | train_acc: 0.8941 | test_loss: 0.5618 | test_acc: 0.8017
Epoch: 8 | train_loss: 0.3002 | train_acc: 0.8876 | test_loss: 0.5536 | test_acc: 0.8110
Epoch: 9 | train_loss: 0.3023 | train_acc: 0.8840 | test_loss: 0.5296 | test_acc: 0.8110
Epoch: 10 | train_loss: 0.2956 | train_acc: 0.8889 | test_loss: 0.6014 | test_acc: 0.7924
Epoch: 11 | train_loss: 0.3043 | train_acc: 0.8885 | test_loss: 0.6313 | test_acc: 0.7870
Epoch: 12 | train_l

{'train_loss': [0.31576487601648834,
  0.32603508020018007,
  0.3234133259888659,
  0.3017820236107982,
  0.33579028125667404,
  0.2831630105456562,
  0.2990685265696218,
  0.30015606921932375,
  0.3022677312400324,
  0.2955837796395975,
  0.304342629876754,
  0.33231339941844873,
  0.3039344231604684,
  0.31510706864138865,
  0.2895400904594584,
  0.30999352280975234,
  0.2705772643420079,
  0.30193349008336134,
  0.29686771076938784,
  0.2982128178766856,
  0.27799257939905986,
  0.27419239763461106,
  0.2701327401272794,
  0.30491550137282264,
  0.2672644583970731,
  0.26858548653569625,
  0.2695609839197169,
  0.29324573249364577,
  0.29717431386523213,
  0.27179865531147795,
  0.27918065714497936,
  0.25642910693501325,
  0.275154987653942,
  0.27358875907164937,
  0.26935460960082974,
  0.2751524147333194,
  0.2833629199006456,
  0.272882665857567,
  0.30717858156942307,
  0.266112236028656,
  0.27071316505894594,
  0.27874423420809685,
  0.2510982328854131,
  0.28291091505200305

Epoch: 45 | train_loss: 0.2473 | train_acc: 0.9091 | test_loss: 0.4972 | test_acc: 0.8190

Epoch: 46 | train_loss: 0.2515 | train_acc: 0.9065 | test_loss: 0.5349 | test_acc: 0.8296

Epoch: 47 | train_loss: 0.2829 | train_acc: 0.9009 | test_loss: 0.5455 | test_acc: 0.8203

Epoch: 48 | train_loss: 0.2674 | train_acc: 0.8944 | test_loss: 0.5817 | test_acc: 0.8030

Epoch: 49 | train_loss: 0.2622 | train_acc: 0.9020 | test_loss: 0.5050 | test_acc: 0.8150

Epoch: 50 | train_loss: 0.2718 | train_acc: 0.9011 | test_loss: 0.6905 | test_acc: 0.7576

## Model 500: 32 hidden units, num_magnitude_bins = 31 and BatchNorm2d

In [ ]:
from torch import nn
class TinyVGG_1_64(nn.Module):
  """
  Model architechture copying TinyVGG from CNN Explainer
  """
  def __init__(self,
               input_shape : int,
               hidden_units : int,
               output_shape : int) -> None:

    super().__init__()

    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels = input_shape,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 0),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.Conv2d(in_channels = hidden_units,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 0),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2,
                     stride = 2)
    )

    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(
            in_channels = hidden_units,
            out_channels = hidden_units,
            kernel_size = 3,
            stride = 1,
            padding = 0
        ),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.Conv2d(in_channels = hidden_units,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 0),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2,
                     stride = 2)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features = 5408,
                  out_features = output_shape)
    )

  def forward(self, x):
    x = self.conv_block_1(x)
    x = self.conv_block_2(x)
    x = self.classifier(x)
    return x

In [ ]:
augmented_transform_2_64 = transforms.Compose([
    transforms.Resize(size = (64, 64)),
    transforms.TrivialAugmentWide(num_magnitude_bins = 31),
    transforms.ToTensor()
])

simple_transform_2_64 = transforms.Compose([
    transforms.Resize(size = (64, 64)),
    transforms.ToTensor()
])

train_data_2_64 = datasets.ImageFolder(
    root = train_dir,
    transform = augmented_transform_2_64,
)

test_data_2_64 = datasets.ImageFolder(
    root = test_dir,
    transform = simple_transform_2_64
)

BATCH_SIZE = 16

train_dataloader_2_64 = DataLoader(
    dataset = train_data_2_64,
    batch_size = BATCH_SIZE,
    shuffle = True,
    num_workers = os.cpu_count()
)

test_dataloader_2_64 = DataLoader(
    dataset = test_data_2_64,
    batch_size = BATCH_SIZE,
    shuffle = False,
    num_workers = os.cpu_count()
)

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model_500 = TinyVGG_1_64(input_shape = 3,
                    hidden_units= 32,
                    output_shape = 3).to(device)

In [ ]:
optimizer_500 = torch.optim.Adam(params = model_500.parameters(),
                             lr = 0.001)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

train(model = model_500,
      train_dataloader = train_dataloader_2_64,
      test_dataloader = test_dataloader_2_64,
      optimizer = optimizer_500,
      loss_fn = loss_fn,
      epochs = 50)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 1.0932 | train_acc: 0.4918 | test_loss: 1.0378 | test_acc: 0.5479
Epoch: 2 | train_loss: 1.0389 | train_acc: 0.5146 | test_loss: 0.8070 | test_acc: 0.6282
Epoch: 3 | train_loss: 0.9479 | train_acc: 0.5593 | test_loss: 1.0637 | test_acc: 0.5260
Epoch: 4 | train_loss: 0.9197 | train_acc: 0.5887 | test_loss: 1.0895 | test_acc: 0.5652
Epoch: 5 | train_loss: 0.8963 | train_acc: 0.5891 | test_loss: 0.7742 | test_acc: 0.6596
Epoch: 6 | train_loss: 0.8499 | train_acc: 0.6233 | test_loss: 0.7192 | test_acc: 0.6928
Epoch: 7 | train_loss: 0.8153 | train_acc: 0.6310 | test_loss: 0.6578 | test_acc: 0.7101
Epoch: 8 | train_loss: 0.7814 | train_acc: 0.6656 | test_loss: 0.7714 | test_acc: 0.6510
Epoch: 9 | train_loss: 0.7712 | train_acc: 0.6647 | test_loss: 0.7943 | test_acc: 0.6634
Epoch: 10 | train_loss: 0.7734 | train_acc: 0.6674 | test_loss: 0.6029 | test_acc: 0.7627
Epoch: 11 | train_loss: 0.7448 | train_acc: 0.6841 | test_loss: 0.6368 | test_acc: 0.7394
Epoch: 12 | train_l

{'train_loss': [1.0932081531971058,
  1.0389173157671665,
  0.9479209406578795,
  0.9196934376625304,
  0.8962699496154244,
  0.8499086421009496,
  0.8152896939439976,
  0.7814474644813132,
  0.7711608805132251,
  0.773357370643751,
  0.7448204521169054,
  0.7175009039276881,
  0.7118228050411171,
  0.6970363916657495,
  0.6856540456308541,
  0.6685425049446999,
  0.6504604979187039,
  0.6320096696099491,
  0.6169505851699951,
  0.6266949185242889,
  0.6193342796454193,
  0.6065804202717247,
  0.5869043327815143,
  0.563101498476157,
  0.5825195957160165,
  0.5763318509074813,
  0.5474864181051863,
  0.5401724394120223,
  0.5334003624763894,
  0.5303009122821456,
  0.5318981596130006,
  0.5215742523577196,
  0.4937896210673853,
  0.5107325178088872,
  0.5072874152596961,
  0.4779398670644625,
  0.48764152888287887,
  0.4671006871664778,
  0.4673839506316692,
  0.4632287729293742,
  0.4447991759219068,
  0.448745552318316,
  0.42972414465026654,
  0.4344431258051108,
  0.430661376169387

Epoch: 45 | train_loss: 0.4307 | train_acc: 0.8295 | test_loss: 0.4387 | test_acc: 0.8271

Epoch: 46 | train_loss: 0.4345 | train_acc: 0.8348 | test_loss: 0.4645 | test_acc: 0.8051

Epoch: 47 | train_loss: 0.4179 | train_acc: 0.8402 | test_loss: 0.4096 | test_acc: 0.8495

Epoch: 48 | train_loss: 0.4233 | train_acc: 0.8355 | test_loss: 0.4326 | test_acc: 0.8399

Epoch: 49 | train_loss: 0.4238 | train_acc: 0.8354 | test_loss: 0.4171 | test_acc: 0.8334

Epoch: 50 | train_loss: 0.4399 | train_acc: 0.8309 | test_loss: 0.4449 | test_acc: 0.8334

In [ ]:
optimizer_500 = torch.optim.Adam(params = model_500.parameters(),
                             lr = 0.0005)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
train(model = model_500,
      train_dataloader = train_dataloader_2_64,
      test_dataloader = test_dataloader_2_64,
      optimizer = optimizer_500,
      loss_fn = loss_fn,
      epochs = 50)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 0.3924 | train_acc: 0.8504 | test_loss: 0.3835 | test_acc: 0.8587
Epoch: 2 | train_loss: 0.3541 | train_acc: 0.8566 | test_loss: 0.4063 | test_acc: 0.8494
Epoch: 3 | train_loss: 0.3699 | train_acc: 0.8581 | test_loss: 0.3657 | test_acc: 0.8547
Epoch: 4 | train_loss: 0.3601 | train_acc: 0.8666 | test_loss: 0.3894 | test_acc: 0.8467
Epoch: 5 | train_loss: 0.3761 | train_acc: 0.8543 | test_loss: 0.4459 | test_acc: 0.8254
Epoch: 6 | train_loss: 0.3510 | train_acc: 0.8652 | test_loss: 0.4810 | test_acc: 0.8197
Epoch: 7 | train_loss: 0.3525 | train_acc: 0.8637 | test_loss: 0.3918 | test_acc: 0.8465
Epoch: 8 | train_loss: 0.3490 | train_acc: 0.8712 | test_loss: 0.4195 | test_acc: 0.8440
Epoch: 9 | train_loss: 0.3358 | train_acc: 0.8756 | test_loss: 0.4161 | test_acc: 0.8452
Epoch: 10 | train_loss: 0.3539 | train_acc: 0.8650 | test_loss: 0.3808 | test_acc: 0.8533
Epoch: 11 | train_loss: 0.3275 | train_acc: 0.8768 | test_loss: 0.3688 | test_acc: 0.8532
Epoch: 12 | train_l

{'train_loss': [0.3923789868857844,
  0.3540862879009112,
  0.3699000606194456,
  0.3600713237063259,
  0.37609966846962345,
  0.3509728386733972,
  0.35247796530841935,
  0.3489636405509837,
  0.3357764241014812,
  0.35390243024056683,
  0.32752796891310537,
  0.33224334121278837,
  0.3394208804089972,
  0.3242922589605582,
  0.29855459862461325,
  0.34334486331922787,
  0.3243164690673774,
  0.331351046285308,
  0.31717747904307453,
  0.2934709056313579,
  0.31351313422969046,
  0.30318476241214054,
  0.3314302457456893,
  0.33608044981163865,
  0.3274680471166651,
  0.33312839045063825,
  0.3078090439927071,
  0.30350733152095305,
  0.284285122673985,
  0.3174085390377552,
  0.32441305155132677,
  0.31480125784345553,
  0.2996243144127917,
  0.29446766509654676,
  0.2992684806280948,
  0.3127503901561524,
  0.29851067721420993,
  0.30409859245022136,
  0.2842891735705078,
  0.28321581196489065,
  0.2910670482467675,
  0.30351110035223317,
  0.3107123688021873,
  0.31450222958381296,

Epoch: 45 | train_loss: 0.2921 | train_acc: 0.8905 | test_loss: 0.4300 | test_acc: 0.8387

Epoch: 46 | train_loss: 0.2843 | train_acc: 0.8946 | test_loss: 0.4462 | test_acc: 0.8361

Epoch: 47 | train_loss: 0.2925 | train_acc: 0.8923 | test_loss: 0.4119 | test_acc: 0.8469

Epoch: 48 | train_loss: 0.2799 | train_acc: 0.8855 | test_loss: 0.3678 | test_acc: 0.8627

Epoch: 49 | train_loss: 0.2890 | train_acc: 0.8907 | test_loss: 0.4258 | test_acc: 0.8400

Epoch: 50 | train_loss: 0.3140 | train_acc: 0.8878 | test_loss: 0.3946 | test_acc: 0.8345

## Model 600: 32 hidden units, num_magnitude_bins = 31 , BatchNorm2d and 96 pixel images


In [ ]:
from torch import nn
class TinyVGG_1_96(nn.Module):
  """
  Model architechture copying TinyVGG from CNN Explainer
  """
  def __init__(self,
               input_shape : int,
               hidden_units : int,
               output_shape : int) -> None:

    super().__init__()

    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels = input_shape,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 0),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.Conv2d(in_channels = hidden_units,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 0),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2,
                     stride = 2)
    )

    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(
            in_channels = hidden_units,
            out_channels = hidden_units,
            kernel_size = 3,
            stride = 1,
            padding = 0
        ),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.Conv2d(in_channels = hidden_units,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 0),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2,
                     stride = 2)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features = 14112,
                  out_features = output_shape)
    )

  def forward(self, x):
    x = self.conv_block_1(x)
    x = self.conv_block_2(x)
    x = self.classifier(x)
    return x

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model_600 = TinyVGG_1_96(input_shape = 3,
                    hidden_units= 32,
                    output_shape = 3).to(device)

In [ ]:
optimizer_600 = torch.optim.Adam(params = model_600.parameters(),
                             lr = 0.001)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

train(model = model_600,
      train_dataloader = train_dataloader_2_96,
      test_dataloader = test_dataloader_2_96,
      optimizer = optimizer_600,
      loss_fn = loss_fn,
      epochs = 50)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 1.6245 | train_acc: 0.4644 | test_loss: 0.9977 | test_acc: 0.5085
Epoch: 2 | train_loss: 1.2097 | train_acc: 0.4884 | test_loss: 0.8756 | test_acc: 0.6041
Epoch: 3 | train_loss: 1.0486 | train_acc: 0.5236 | test_loss: 0.8643 | test_acc: 0.5963
Epoch: 4 | train_loss: 0.9744 | train_acc: 0.5413 | test_loss: 0.8159 | test_acc: 0.6299
Epoch: 5 | train_loss: 0.9807 | train_acc: 0.5549 | test_loss: 0.7478 | test_acc: 0.6807
Epoch: 6 | train_loss: 0.9250 | train_acc: 0.5849 | test_loss: 0.7283 | test_acc: 0.6960
Epoch: 7 | train_loss: 0.8785 | train_acc: 0.6059 | test_loss: 0.7842 | test_acc: 0.6689
Epoch: 8 | train_loss: 0.8742 | train_acc: 0.6091 | test_loss: 0.6722 | test_acc: 0.7147
Epoch: 9 | train_loss: 0.8196 | train_acc: 0.6425 | test_loss: 0.6980 | test_acc: 0.7107
Epoch: 10 | train_loss: 0.8061 | train_acc: 0.6417 | test_loss: 0.6342 | test_acc: 0.7428
Epoch: 11 | train_loss: 0.7779 | train_acc: 0.6585 | test_loss: 0.6844 | test_acc: 0.7126
Epoch: 12 | train_l

{'train_loss': [1.6245439830401265,
  1.2097414233160357,
  1.0485637868549806,
  0.9743851357740714,
  0.9807262038085478,
  0.9249504284655794,
  0.8785287196754564,
  0.8741945499646748,
  0.8196034581525952,
  0.8060548299170555,
  0.7779353068652728,
  0.7508795542497162,
  0.7515753372341183,
  0.7256466950296511,
  0.7274854688356954,
  0.6984510510525805,
  0.6766390758203277,
  0.6681573460710809,
  0.6670882302395841,
  0.662675881428076,
  0.6294519966798471,
  0.6334469762254269,
  0.6145412544620797,
  0.5987120922786969,
  0.5893459717432658,
  0.5891266228459405,
  0.5689440225878506,
  0.5567675398807999,
  0.5634429162064343,
  0.5408558138507478,
  0.5391027025931271,
  0.5046587479664079,
  0.516058161220652,
  0.49536873151858646,
  0.4586329611269295,
  0.48780379169587545,
  0.4851644841280389,
  0.4654347889706598,
  0.4744349066035967,
  0.47187908923795036,
  0.4617557239764971,
  0.44391044747110797,
  0.4230921316548442,
  0.41234962016027027,
  0.42898555699

Epoch: 45 | train_loss: 0.4290 | train_acc: 0.8316 | test_loss: 0.5268 | test_acc: 0.8081

Epoch: 46 | train_loss: 0.4449 | train_acc: 0.8317 | test_loss: 0.4634 | test_acc: 0.8134

Epoch: 47 | train_loss: 0.3846 | train_acc: 0.8506 | test_loss: 0.4955 | test_acc: 0.8186

Epoch: 48 | train_loss: 0.4186 | train_acc: 0.8402 | test_loss: 0.5333 | test_acc: 0.7829

Epoch: 49 | train_loss: 0.3924 | train_acc: 0.8468 | test_loss: 0.6399 | test_acc: 0.7620

Epoch: 50 | train_loss: 0.3943 | train_acc: 0.8529 | test_loss: 0.4986 | test_acc: 0.8176

## Model 700: 64 hidden units, num_magnitude_bins = 31 , BatchNorm2d , 96 pixel images

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model_700 = TinyVGG_1_96(input_shape = 3,
                    hidden_units= 64,
                    output_shape = 3).to(device)

In [ ]:
optimizer_700 = torch.optim.Adam(params = model_700.parameters(),
                             lr = 0.001)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

train(model = model_700,
      train_dataloader = train_dataloader_2_96,
      test_dataloader = test_dataloader_2_96,
      optimizer = optimizer_700,
      loss_fn = loss_fn,
      epochs = 50)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 1.8276 | train_acc: 0.4609 | test_loss: 1.3379 | test_acc: 0.5133
Epoch: 2 | train_loss: 1.2443 | train_acc: 0.4786 | test_loss: 1.3441 | test_acc: 0.4816
Epoch: 3 | train_loss: 1.0676 | train_acc: 0.5330 | test_loss: 0.8281 | test_acc: 0.6305
Epoch: 4 | train_loss: 1.1173 | train_acc: 0.5278 | test_loss: 0.9161 | test_acc: 0.5785
Epoch: 5 | train_loss: 1.0148 | train_acc: 0.5501 | test_loss: 0.7819 | test_acc: 0.6436
Epoch: 6 | train_loss: 0.9377 | train_acc: 0.5919 | test_loss: 0.7255 | test_acc: 0.6877
Epoch: 7 | train_loss: 0.8719 | train_acc: 0.6116 | test_loss: 0.7716 | test_acc: 0.6556
Epoch: 8 | train_loss: 0.8609 | train_acc: 0.6151 | test_loss: 0.7419 | test_acc: 0.6873
Epoch: 9 | train_loss: 0.8478 | train_acc: 0.6146 | test_loss: 1.0603 | test_acc: 0.5559
Epoch: 10 | train_loss: 0.8163 | train_acc: 0.6387 | test_loss: 0.7600 | test_acc: 0.6700
Epoch: 11 | train_loss: 0.8121 | train_acc: 0.6443 | test_loss: 0.7570 | test_acc: 0.6810
Epoch: 12 | train_l

{'train_loss': [1.827637697365267,
  1.2442504852376086,
  1.0675954499565963,
  1.1173306308316846,
  1.0147804770063846,
  0.9376917242581118,
  0.871946984994496,
  0.8608814909103069,
  0.8478264931245898,
  0.8163320825877765,
  0.8120504605009201,
  0.7675923770623849,
  0.7607090542925165,
  0.7762304726221883,
  0.7324295749901034,
  0.70266440891205,
  0.6758308292280698,
  0.6723796869000644,
  0.6759345077031048,
  0.6393626381953558,
  0.6253895668696005,
  0.6084867753881089,
  0.5810278013665625,
  0.561643325902046,
  0.5749780834777981,
  0.5436944781360051,
  0.5439435897778112,
  0.5263731536713052,
  0.5045199633067381,
  0.4806787040850795,
  0.5115582373865107,
  0.4465624969584722,
  0.46931399605798385,
  0.4665342536061368,
  0.4513847813380103,
  0.4390920995080725,
  0.4314214722484562,
  0.40541038080944236,
  0.4489923699739132,
  0.42029537006895595,
  0.44001542808527644,
  0.3932624585820851,
  0.40245557727014764,
  0.3823388052694764,
  0.34784714259365

Epoch: 45 | train_loss: 0.3478 | train_acc: 0.8723 | test_loss: 0.4644 | test_acc: 0.8222

Epoch: 46 | train_loss: 0.3825 | train_acc: 0.8590 | test_loss: 0.3846 | test_acc: 0.8494

Epoch: 47 | train_loss: 0.3870 | train_acc: 0.8539 | test_loss: 0.7248 | test_acc: 0.7720

Epoch: 48 | train_loss: 0.3609 | train_acc: 0.8622 | test_loss: 0.4679 | test_acc: 0.8133

Epoch: 49 | train_loss: 0.3561 | train_acc: 0.8628 | test_loss: 0.3996 | test_acc: 0.8560

Epoch: 50 | train_loss: 0.3640 | train_acc: 0.8696 | test_loss: 0.3988 | test_acc: 0.8465

In [ ]:
optimizer_700 = torch.optim.Adam(params = model_700.parameters(),
                             lr = 0.0005)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
train(model = model_700,
      train_dataloader = train_dataloader_2_96,
      test_dataloader = test_dataloader_2_96,
      optimizer = optimizer_700,
      loss_fn = loss_fn,
      epochs = 50)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 0.3005 | train_acc: 0.8811 | test_loss: 0.4635 | test_acc: 0.8254
Epoch: 2 | train_loss: 0.3042 | train_acc: 0.8815 | test_loss: 0.4322 | test_acc: 0.8267
Epoch: 3 | train_loss: 0.2920 | train_acc: 0.8967 | test_loss: 0.3935 | test_acc: 0.8429
Epoch: 4 | train_loss: 0.2918 | train_acc: 0.8960 | test_loss: 0.3896 | test_acc: 0.8507
Epoch: 5 | train_loss: 0.2837 | train_acc: 0.8929 | test_loss: 0.4126 | test_acc: 0.8560
Epoch: 6 | train_loss: 0.2870 | train_acc: 0.8940 | test_loss: 0.3464 | test_acc: 0.8682
Epoch: 7 | train_loss: 0.2632 | train_acc: 0.9062 | test_loss: 0.4457 | test_acc: 0.8385
Epoch: 8 | train_loss: 0.2963 | train_acc: 0.8920 | test_loss: 0.3412 | test_acc: 0.8735
Epoch: 9 | train_loss: 0.2914 | train_acc: 0.9024 | test_loss: 0.4441 | test_acc: 0.8343
Epoch: 10 | train_loss: 0.2742 | train_acc: 0.8972 | test_loss: 0.3740 | test_acc: 0.8454
Epoch: 11 | train_loss: 0.2731 | train_acc: 0.9000 | test_loss: 0.3555 | test_acc: 0.8708
Epoch: 12 | train_l

{'train_loss': [0.3004952355268154,
  0.3042009147346442,
  0.2919988339727229,
  0.2918078776528227,
  0.2836562314909612,
  0.28704685715179074,
  0.2631589018653893,
  0.2962592710205849,
  0.2913913247010387,
  0.2741737986075963,
  0.27311937333949915,
  0.27945977087456286,
  0.25460102672008333,
  0.2763812583706058,
  0.29068513933225726,
  0.2693150124532428,
  0.24857862194643376,
  0.2535509157476696,
  0.2662493850481003,
  0.2695450951999172,
  0.24160169461306105,
  0.2526544694729308,
  0.2333317707418233,
  0.26044463211049634,
  0.23135687103032643,
  0.22754939736679514,
  0.25309592308410517,
  0.2733434187974913,
  0.2468583908341561,
  0.23782357123039716,
  0.21915878562903995,
  0.23913719868353495,
  0.24945395593102096,
  0.21998140460923843,
  0.23596308150507034,
  0.25493248227111837,
  0.22998271031793974,
  0.23588737639340948,
  0.2198509048168541,
  0.23673946795496323,
  0.2313358963909724,
  0.2370605449099754,
  0.2188732409585558,
  0.228460192234512

Epoch: 45 | train_loss: 0.2181 | train_acc: 0.9220 | test_loss: 0.3924 | test_acc: 0.8560

Epoch: 46 | train_loss: 0.2298 | train_acc: 0.9188 | test_loss: 0.3704 | test_acc: 0.8627

Epoch: 47 | train_loss: 0.2231 | train_acc: 0.9229 | test_loss: 0.3729 | test_acc: 0.8642

Epoch: 48 | train_loss: 0.2179 | train_acc: 0.9204 | test_loss: 0.3322 | test_acc: 0.8892

Epoch: 49 | train_loss: 0.2210 | train_acc: 0.9127 | test_loss: 0.5582 | test_acc: 0.8169

Epoch: 50 | train_loss: 0.2017 | train_acc: 0.9270 | test_loss: 0.3538 | test_acc: 0.8665

## Model 800: 64 hidden units, num_magnitude_bins = 31 , BatchNorm2d , 128 pixel images

In [ ]:
from torch import nn
class TinyVGG_1_128(nn.Module):
  """
  Model architechture copying TinyVGG from CNN Explainer
  """
  def __init__(self,
               input_shape : int,
               hidden_units : int,
               output_shape : int) -> None:

    super().__init__()

    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels = input_shape,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 0),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.Conv2d(in_channels = hidden_units,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 0),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2,
                     stride = 2)
    )

    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(
            in_channels = hidden_units,
            out_channels = hidden_units,
            kernel_size = 3,
            stride = 1,
            padding = 0
        ),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.Conv2d(in_channels = hidden_units,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 0),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2,
                     stride = 2)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features = 841 * hidden_units,
                  out_features = output_shape)
    )

  def forward(self, x):
    x = self.conv_block_1(x)
    x = self.conv_block_2(x)
    x = self.classifier(x)
    return x

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model_800 = TinyVGG_1_128(input_shape = 3,
                    hidden_units= 64,
                    output_shape = 3).to(device)

In [ ]:
optimizer_800 = torch.optim.Adam(params = model_800.parameters(),
                             lr = 0.001)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

train(model = model_800,
      train_dataloader = train_dataloader_2_128,
      test_dataloader = test_dataloader_2_128,
      optimizer = optimizer_800,
      loss_fn = loss_fn,
      epochs = 50)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 3.4160 | train_acc: 0.4401 | test_loss: 2.0667 | test_acc: 0.4280
Epoch: 2 | train_loss: 1.4922 | train_acc: 0.5004 | test_loss: 0.9941 | test_acc: 0.5849
Epoch: 3 | train_loss: 1.2891 | train_acc: 0.4807 | test_loss: 0.9452 | test_acc: 0.5530
Epoch: 4 | train_loss: 1.0396 | train_acc: 0.5370 | test_loss: 1.0801 | test_acc: 0.5534
Epoch: 5 | train_loss: 0.9762 | train_acc: 0.5623 | test_loss: 0.8157 | test_acc: 0.6484
Epoch: 6 | train_loss: 0.9710 | train_acc: 0.5655 | test_loss: 0.8234 | test_acc: 0.6366
Epoch: 7 | train_loss: 0.8852 | train_acc: 0.5883 | test_loss: 0.7773 | test_acc: 0.6537
Epoch: 8 | train_loss: 0.8570 | train_acc: 0.6229 | test_loss: 0.7365 | test_acc: 0.6890
Epoch: 9 | train_loss: 0.8755 | train_acc: 0.6111 | test_loss: 0.7344 | test_acc: 0.6905
Epoch: 10 | train_loss: 0.8298 | train_acc: 0.6293 | test_loss: 0.6582 | test_acc: 0.7232
Epoch: 11 | train_loss: 0.7780 | train_acc: 0.6610 | test_loss: 0.7078 | test_acc: 0.6943
Epoch: 12 | train_l

{'train_loss': [3.416033745657468,
  1.492159384784969,
  1.2890674962219617,
  1.0395634339210835,
  0.9761901607750155,
  0.9709616265398391,
  0.8852175152893608,
  0.8569734035231543,
  0.8755401032613525,
  0.8297944525454907,
  0.7780162840024799,
  0.7850397605422541,
  0.7679773173856397,
  0.7037033728251221,
  0.7429952623573601,
  0.7321201626290667,
  0.6874551725514392,
  0.6850224340215643,
  0.6461848678952413,
  0.6298601121133101,
  0.6117817592536304,
  0.6075032218128231,
  0.5635109209845252,
  0.5589217095510334,
  0.5506224635433643,
  0.49932075043519336,
  0.5478821258384285,
  0.49791305041904993,
  0.46254846659746574,
  0.4866404001805799,
  0.4560082473442064,
  0.43956453474700874,
  0.41086358592865313,
  0.43869113285385125,
  0.4018486834375571,
  0.44480541049905703,
  0.3776218114909551,
  0.40559257577497065,
  0.38864372290195304,
  0.37182214504755134,
  0.3780430002098388,
  0.3802760873544723,
  0.37145854796923644,
  0.38405373306773233,
  0.3652

Epoch: 45 | train_loss: 0.3652 | train_acc: 0.8679 | test_loss: 0.4650 | test_acc: 0.8129

Epoch: 46 | train_loss: 0.3619 | train_acc: 0.8687 | test_loss: 0.4627 | test_acc: 0.8279

Epoch: 47 | train_loss: 0.3541 | train_acc: 0.8687 | test_loss: 0.5018 | test_acc: 0.7998

Epoch: 48 | train_loss: 0.3670 | train_acc: 0.8579 | test_loss: 0.4782 | test_acc: 0.8131

Epoch: 49 | train_loss: 0.3286 | train_acc: 0.8816 | test_loss: 0.6162 | test_acc: 0.7863

Epoch: 50 | train_loss: 0.3678 | train_acc: 0.8597 | test_loss: 0.5302 | test_acc: 0.8026

In [ ]:
optimizer_800 = torch.optim.Adam(params = model_800.parameters(),
                             lr = 0.0005)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
train(model = model_800,
      train_dataloader = train_dataloader_2_128,
      test_dataloader = test_dataloader_2_128,
      optimizer = optimizer_800,
      loss_fn = loss_fn,
      epochs = 50)

## Model 900: 128 hidden units, data num_magnitude_bins = 31 , BatchNorm2d , 128 pixel images

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model_900 = TinyVGG_1_128(input_shape = 3,
                    hidden_units= 128,
                    output_shape = 3).to(device)

In [ ]:
optimizer_900 = torch.optim.Adam(params = model_900.parameters(),
                             lr = 0.001)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

train(model = model_900,
      train_dataloader = train_dataloader_2_128,
      test_dataloader = test_dataloader_2_128,
      optimizer = optimizer_900,
      loss_fn = loss_fn,
      epochs = 50)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 8.6319 | train_acc: 0.4262 | test_loss: 4.9034 | test_acc: 0.3737
Epoch: 2 | train_loss: 3.3904 | train_acc: 0.4546 | test_loss: 1.4778 | test_acc: 0.5378
Epoch: 3 | train_loss: 2.0594 | train_acc: 0.4976 | test_loss: 2.4476 | test_acc: 0.4016
Epoch: 4 | train_loss: 1.4219 | train_acc: 0.5287 | test_loss: 0.9381 | test_acc: 0.6157
Epoch: 5 | train_loss: 1.2767 | train_acc: 0.5337 | test_loss: 1.0366 | test_acc: 0.6033
Epoch: 6 | train_loss: 1.0773 | train_acc: 0.5417 | test_loss: 0.8343 | test_acc: 0.6373
Epoch: 7 | train_loss: 1.0025 | train_acc: 0.5699 | test_loss: 0.8190 | test_acc: 0.6381
Epoch: 8 | train_loss: 0.9839 | train_acc: 0.5699 | test_loss: 0.8058 | test_acc: 0.6436
Epoch: 9 | train_loss: 0.8853 | train_acc: 0.5963 | test_loss: 0.9405 | test_acc: 0.5737
Epoch: 10 | train_loss: 0.8676 | train_acc: 0.6147 | test_loss: 0.8657 | test_acc: 0.6434
Epoch: 11 | train_loss: 0.8387 | train_acc: 0.6251 | test_loss: 0.7574 | test_acc: 0.6829
Epoch: 12 | train_l

{'train_loss': [8.631912551027662,
  3.3904283959814845,
  2.059443724916336,
  1.4219402734269486,
  1.2766993100761521,
  1.0772547083543547,
  1.002513052935296,
  0.9838800419729652,
  0.8853334535098245,
  0.8676452898810095,
  0.8386558624024086,
  0.7815138289691709,
  0.8165371821704486,
  0.7628926023946586,
  0.7495925263732883,
  0.7265776461320566,
  0.7303577999696664,
  0.7018194281040354,
  0.6778035119492957,
  0.635793241837346,
  0.6746526661705463,
  0.6491364905386107,
  0.645143930687972,
  0.5955370662482917,
  0.583632156147179,
  0.5419347242894748,
  0.5443532474286167,
  0.558222161856949,
  0.5589221113539756,
  0.5311393914281899,
  0.5033836303450537,
  0.5073188255864678,
  0.4687481718600219,
  0.44156695881211167,
  0.4656999828756278,
  0.46094119242319825,
  0.44529160418620345,
  0.4516591931699861,
  0.42242169628540677,
  0.44498631823147444,
  0.4052336903434273,
  0.40367665789123125,
  0.3901168530714427,
  0.37658226978799975,
  0.37312048245617

Epoch: 45 | train_loss: 0.3731 | train_acc: 0.8656 | test_loss: 0.4321 | test_acc: 0.8376

Epoch: 46 | train_loss: 0.3921 | train_acc: 0.8497 | test_loss: 0.4870 | test_acc: 0.8013

Epoch: 47 | train_loss: 0.3861 | train_acc: 0.8594 | test_loss: 0.4295 | test_acc: 0.8362

Epoch: 48 | train_loss: 0.3860 | train_acc: 0.8598 | test_loss: 0.4660 | test_acc: 0.8267

Epoch: 49 | train_loss: 0.3638 | train_acc: 0.8672 | test_loss: 0.4524 | test_acc: 0.8361

Epoch: 50 | train_loss: 0.3584 | train_acc: 0.8687 | test_loss: 0.5084 | test_acc: 0.8231

In [ ]:
optimizer_900 = torch.optim.Adam(params = model_900.parameters(),
                             lr = 0.0005)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
train(model = model_900,
      train_dataloader = train_dataloader_2_128,
      test_dataloader = test_dataloader_2_128,
      optimizer = optimizer_900,
      loss_fn = loss_fn,
      epochs = 50)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 0.3405 | train_acc: 0.8724 | test_loss: 0.4210 | test_acc: 0.8457
Epoch: 2 | train_loss: 0.3049 | train_acc: 0.8883 | test_loss: 0.4709 | test_acc: 0.8319
Epoch: 3 | train_loss: 0.3275 | train_acc: 0.8778 | test_loss: 0.4529 | test_acc: 0.8347
Epoch: 4 | train_loss: 0.2899 | train_acc: 0.8918 | test_loss: 0.4618 | test_acc: 0.8349
Epoch: 5 | train_loss: 0.2747 | train_acc: 0.8972 | test_loss: 0.4579 | test_acc: 0.8467
Epoch: 6 | train_loss: 0.3066 | train_acc: 0.8895 | test_loss: 0.3981 | test_acc: 0.8682
Epoch: 7 | train_loss: 0.2822 | train_acc: 0.8932 | test_loss: 0.4558 | test_acc: 0.8385
Epoch: 8 | train_loss: 0.2734 | train_acc: 0.9028 | test_loss: 0.5020 | test_acc: 0.8429
Epoch: 9 | train_loss: 0.2632 | train_acc: 0.9082 | test_loss: 0.4793 | test_acc: 0.8319
Epoch: 10 | train_loss: 0.2795 | train_acc: 0.8980 | test_loss: 0.5198 | test_acc: 0.8064
Epoch: 11 | train_loss: 0.2683 | train_acc: 0.9035 | test_loss: 0.4063 | test_acc: 0.8520
Epoch: 12 | train_l

{'train_loss': [0.34047252896195607,
  0.30490675425909936,
  0.3274802450381272,
  0.2898755662988686,
  0.27471855708153536,
  0.3066283586428415,
  0.28216668267252176,
  0.2733732041314984,
  0.26320397018963565,
  0.27951555467270156,
  0.2683274963546993,
  0.26150025208051325,
  0.24967329822157028,
  0.291625985888936,
  0.24246306820435726,
  0.2461476411062775,
  0.2683158943574902,
  0.26429797724840487,
  0.21871128728520786,
  0.2566389473194772,
  0.24998749132063372,
  0.26848309958710315,
  0.24691325238516146,
  0.21587532403026807,
  0.2222170221573072,
  0.23701943231231354,
  0.23810852775416264,
  0.21436655426279028,
  0.2138946190714202,
  0.25250412808097106,
  0.22701828276857416,
  0.23298053695747617,
  0.22645878473442074,
  0.219589461981986,
  0.20812884669301782,
  0.22940290176329461,
  0.2454055091168018,
  0.22424137782543263,
  0.22835888818619735,
  0.22316665004225486,
  0.21381376535096702,
  0.23227721184544953,
  0.23949997661740963,
  0.18965004

Epoch: 44 | train_loss: 0.1897 | train_acc: 0.9332 | test_loss: 0.4821 | test_acc: 0.8294

Epoch: 45 | train_loss: 0.2065 | train_acc: 0.9255 | test_loss: 0.3697 | test_acc: 0.8684

Epoch: 46 | train_loss: 0.2023 | train_acc: 0.9330 | test_loss: 0.3615 | test_acc: 0.8723

Epoch: 47 | train_loss: 0.2122 | train_acc: 0.9242 | test_loss: 0.4026 | test_acc: 0.8596

Epoch: 48 | train_loss: 0.1942 | train_acc: 0.9317 | test_loss: 0.4188 | test_acc: 0.8385

Epoch: 49 | train_loss: 0.1865 | train_acc: 0.9341 | test_loss: 0.3738 | test_acc: 0.8522

Epoch: 50 | train_loss: 0.2004 | train_acc: 0.9291 | test_loss: 0.4042 | test_acc: 0.8522

## Model 1000: 96 hidden units, num_magnitude_bins = 31 , BatchNorm2d , 96 pixel images

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model_1000 = TinyVGG_1_96(input_shape = 3,
                    hidden_units= 96,
                    output_shape = 3).to(device)

In [ ]:
optimizer_1000 = torch.optim.Adam(params = model_1000.parameters(),
                             lr = 0.001)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

train(model = model_1000,
      train_dataloader = train_dataloader_2_96,
      test_dataloader = test_dataloader_2_96,
      optimizer = optimizer_1000,
      loss_fn = loss_fn,
      epochs = 50)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 3.5720 | train_acc: 0.4322 | test_loss: 1.3607 | test_acc: 0.5386
Epoch: 2 | train_loss: 1.5532 | train_acc: 0.4721 | test_loss: 1.2783 | test_acc: 0.5232
Epoch: 3 | train_loss: 1.2929 | train_acc: 0.4934 | test_loss: 1.0761 | test_acc: 0.4979
Epoch: 4 | train_loss: 1.1242 | train_acc: 0.5032 | test_loss: 0.8226 | test_acc: 0.6345
Epoch: 5 | train_loss: 1.0880 | train_acc: 0.5387 | test_loss: 0.8424 | test_acc: 0.6351
Epoch: 6 | train_loss: 0.9481 | train_acc: 0.5817 | test_loss: 0.8357 | test_acc: 0.6161
Epoch: 7 | train_loss: 0.9131 | train_acc: 0.5934 | test_loss: 0.7183 | test_acc: 0.6924
Epoch: 8 | train_loss: 0.8722 | train_acc: 0.6118 | test_loss: 0.7688 | test_acc: 0.6873
Epoch: 9 | train_loss: 0.8460 | train_acc: 0.6215 | test_loss: 0.6813 | test_acc: 0.7221
Epoch: 10 | train_loss: 0.8156 | train_acc: 0.6413 | test_loss: 0.6607 | test_acc: 0.7264
Epoch: 11 | train_loss: 0.8348 | train_acc: 0.6337 | test_loss: 0.7518 | test_acc: 0.6790
Epoch: 12 | train_l

{'train_loss': [3.5720287979917322,
  1.5532386154993205,
  1.2929294422163184,
  1.1242178272693715,
  1.0880374012263954,
  0.9480531841305131,
  0.9130559677773333,
  0.8721807294703544,
  0.8459734147322093,
  0.8156394926791496,
  0.8347514585611668,
  0.8048282398822459,
  0.767719363278531,
  0.7873195908593793,
  0.7509756177029712,
  0.7428906331671045,
  0.7103071596394194,
  0.7106360528908723,
  0.6965588548927443,
  0.6656359247070678,
  0.6526508721265387,
  0.6295785321622875,
  0.6293228515919219,
  0.619026909060512,
  0.5800189439286577,
  0.5894152958553733,
  0.558915912048191,
  0.5425056232839611,
  0.5378675401633513,
  0.5136910536399124,
  0.5140683690072797,
  0.5386420907915062,
  0.48336533303801893,
  0.45796110158693704,
  0.48053713370088147,
  0.47078993964068433,
  0.4123185032859762,
  0.42948878535353546,
  0.41571225231209546,
  0.41910088691094244,
  0.3982460486656385,
  0.39658983051776886,
  0.4050526951855801,
  0.37409800536772037,
  0.39976451

Epoch: 45 | train_loss: 0.3998 | train_acc: 0.8433 | test_loss: 0.4596 | test_acc: 0.8370

Epoch: 46 | train_loss: 0.3894 | train_acc: 0.8555 | test_loss: 0.4198 | test_acc: 0.8402

Epoch: 47 | train_loss: 0.3859 | train_acc: 0.8535 | test_loss: 0.4824 | test_acc: 0.8429

Epoch: 48 | train_loss: 0.4130 | train_acc: 0.8536 | test_loss: 0.4695 | test_acc: 0.8264

Epoch: 49 | train_loss: 0.3728 | train_acc: 0.8610 | test_loss: 0.5834 | test_acc: 0.7880

Epoch: 50 | train_loss: 0.3499 | train_acc: 0.8621 | test_loss: 0.4537 | test_acc: 0.8317

In [ ]:
optimizer_1000 = torch.optim.Adam(params = model_1000.parameters(),
                             lr = 0.0005)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
train(model = model_1000,
      train_dataloader = train_dataloader_2_96,
      test_dataloader = test_dataloader_2_96,
      optimizer = optimizer_1000,
      loss_fn = loss_fn,
      epochs = 50)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 0.3095 | train_acc: 0.8879 | test_loss: 0.4308 | test_acc: 0.8492
Epoch: 2 | train_loss: 0.3117 | train_acc: 0.8861 | test_loss: 0.5451 | test_acc: 0.8076
Epoch: 3 | train_loss: 0.3022 | train_acc: 0.8922 | test_loss: 0.4402 | test_acc: 0.8307
Epoch: 4 | train_loss: 0.2892 | train_acc: 0.9012 | test_loss: 0.3990 | test_acc: 0.8532
Epoch: 5 | train_loss: 0.2690 | train_acc: 0.8949 | test_loss: 0.3920 | test_acc: 0.8556
Epoch: 6 | train_loss: 0.2687 | train_acc: 0.9020 | test_loss: 0.4271 | test_acc: 0.8532
Epoch: 7 | train_loss: 0.2665 | train_acc: 0.9120 | test_loss: 0.4045 | test_acc: 0.8520
Epoch: 8 | train_loss: 0.2965 | train_acc: 0.8958 | test_loss: 0.4502 | test_acc: 0.8410
Epoch: 9 | train_loss: 0.2794 | train_acc: 0.8925 | test_loss: 0.4366 | test_acc: 0.8412
Epoch: 10 | train_loss: 0.2582 | train_acc: 0.9038 | test_loss: 0.5043 | test_acc: 0.8304
Epoch: 11 | train_loss: 0.2590 | train_acc: 0.9025 | test_loss: 0.4596 | test_acc: 0.8343
Epoch: 12 | train_l

{'train_loss': [0.309503820018037,
  0.31173433140514695,
  0.3021754766451129,
  0.28922506417841354,
  0.2690129944565871,
  0.26870360365467716,
  0.26649855279383505,
  0.29646099305956075,
  0.2793909439962384,
  0.258150837711118,
  0.25900429768606703,
  0.2687021588232923,
  0.2497086616114099,
  0.2565390377880094,
  0.28766824729106527,
  0.25051861770239703,
  0.24494748385557047,
  0.27780644297071383,
  0.29568865298168034,
  0.2521595997035715,
  0.26187983150626964,
  0.2641162611015722,
  0.23275649342818042,
  0.24841468696687238,
  0.2559153854662011,
  0.2377469742815968,
  0.25560687977741375,
  0.2228885850545468,
  0.24756115829532452,
  0.24141489228907417,
  0.23328678472733771,
  0.21372436804710127,
  0.23386683934599373,
  0.23199764610076626,
  0.22421015040777253,
  0.20331745610592214,
  0.23621778701372603,
  0.22031915649533906,
  0.2101651968645817,
  0.22289725769879232,
  0.2220964477371081,
  0.22537217430559034,
  0.21610351262204613,
  0.2239038422

Epoch: 45 | train_loss: 0.2335 | train_acc: 0.9170 | test_loss: 0.4309 | test_acc: 0.8554

Epoch: 46 | train_loss: 0.2009 | train_acc: 0.9257 | test_loss: 0.4489 | test_acc: 0.8492

Epoch: 47 | train_loss: 0.2134 | train_acc: 0.9207 | test_loss: 0.4640 | test_acc: 0.8454

Epoch: 48 | train_loss: 0.2079 | train_acc: 0.9233 | test_loss: 0.3962 | test_acc: 0.8708

Epoch: 49 | train_loss: 0.2299 | train_acc: 0.9207 | test_loss: 0.3545 | test_acc: 0.8744

Epoch: 50 | train_loss: 0.1874 | train_acc: 0.9266 | test_loss: 0.3952 | test_acc: 0.8627

## Model 1100:  num_magnitude_bins = 31 , BatchNorm2d , 96 pixel images, New model structure

In [ ]:
from torch import nn
class TinyVGG_2_96(nn.Module):
  """
  Model architechture copying TinyVGG from CNN Explainer
  """
  def __init__(self,
               input_shape : int,
               hidden_units : int,
               output_shape : int) -> None:

    super().__init__()

    self.conv_block_1 = nn.Sequential(
    nn.Conv2d(3, 32, 3, padding=1, bias=False),
    nn.BatchNorm2d(32),
    nn.ReLU(),

    nn.Conv2d(32, 32, 3, padding=1, bias=False),
    nn.BatchNorm2d(32),
    nn.ReLU(),

    nn.MaxPool2d(2)
    )

    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(32, 64, 3, padding=1, bias=False),
        nn.BatchNorm2d(64),
        nn.ReLU(),

        nn.Conv2d(64, 64, 3, padding=1, bias=False),
        nn.BatchNorm2d(64),
        nn.ReLU(),

        nn.MaxPool2d(2)
    )

    self.conv_block_3 = nn.Sequential(
        nn.Conv2d(64, 128, 3, padding=1, bias=False),
        nn.BatchNorm2d(128),
        nn.ReLU(),

        nn.Conv2d(128, 128, 3, padding=1, bias=False),
        nn.BatchNorm2d(128),
        nn.ReLU(),

        nn.MaxPool2d(2)
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features = 18432,
                  out_features = output_shape)
    )

  def forward(self, x):
    x = self.conv_block_1(x)
    x = self.conv_block_2(x)
    x = self.conv_block_3(x)
    x = self.classifier(x)
    return x

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model_1100 = TinyVGG_2_96(input_shape = 3,
                          hidden_units = 64,
                          output_shape = 3).to(device)

In [ ]:
optimizer_1100 = torch.optim.Adam(params = model_1100.parameters(),
                             lr = 0.001)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
train_data_2_96.class_to_idx

{'pizza': 0, 'steak': 1, 'sushi': 2}

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

train(model = model_1100,
      train_dataloader = train_dataloader_2_96,
      test_dataloader = test_dataloader_2_96,
      optimizer = optimizer_1100,
      loss_fn = loss_fn,
      epochs = 50)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 2.0374 | train_acc: 0.4139 | test_loss: 1.2664 | test_acc: 0.5302
Epoch: 2 | train_loss: 1.2912 | train_acc: 0.4970 | test_loss: 0.7863 | test_acc: 0.6417
Epoch: 3 | train_loss: 1.0280 | train_acc: 0.5228 | test_loss: 0.9583 | test_acc: 0.5467
Epoch: 4 | train_loss: 0.9178 | train_acc: 0.5765 | test_loss: 0.7428 | test_acc: 0.6666
Epoch: 5 | train_loss: 0.9197 | train_acc: 0.5898 | test_loss: 0.7378 | test_acc: 0.6755
New high: 0.7051671732522797, Saving model to : models/model_70-51.pth
Epoch: 6 | train_loss: 0.8830 | train_acc: 0.6038 | test_loss: 0.6940 | test_acc: 0.7052
Epoch: 7 | train_loss: 0.8615 | train_acc: 0.6255 | test_loss: 0.7823 | test_acc: 0.6351
New high: 0.7410714285714286, Saving model to : models/model_74-10.pth
Epoch: 8 | train_loss: 0.8496 | train_acc: 0.6189 | test_loss: 0.6459 | test_acc: 0.7411
Epoch: 9 | train_loss: 0.8146 | train_acc: 0.6499 | test_loss: 0.6707 | test_acc: 0.7225
New high: 0.7532294832826748, Saving model to : models/mo

{'train_loss': [2.0373781006386937,
  1.2912281603677898,
  1.0279693286469642,
  0.9177953977111384,
  0.9197028625941446,
  0.8830148512167288,
  0.861535038719786,
  0.8495999359069987,
  0.8145564581062776,
  0.7774781005602356,
  0.7937901859165083,
  0.8224851072680021,
  0.7501291049287674,
  0.7205591225032265,
  0.7154104253079029,
  0.7169569037484784,
  0.6676899961968685,
  0.6860592292344316,
  0.6602459239198807,
  0.6589613575884636,
  0.6460640639700788,
  0.6142483232080513,
  0.6181612564316878,
  0.6002939611884719,
  0.5986726111765449,
  0.6202418953180313,
  0.5745283853500447,
  0.5546786660420979,
  0.54504945335236,
  0.535778432767442,
  0.531780795319706,
  0.5270744221853026,
  0.48516531204078217,
  0.4825046737777426,
  0.48741706170088855,
  0.4870337338735026,
  0.464816893140475,
  0.4542581653552698,
  0.4652718169257996,
  0.44500079074649945,
  0.4494360643075713,
  0.4262045465991007,
  0.4332789496010077,
  0.387737281541241,
  0.40752057086491417,

Epoch: 45 | train_loss: 0.3857 | train_acc: 0.8519 | test_loss: 0.4211 | test_acc: 0.8279

Epoch: 46 | train_loss: 0.3684 | train_acc: 0.8608 | test_loss: 0.3270 | test_acc: 0.8854

Epoch: 47 | train_loss: 0.3505 | train_acc: 0.8648 | test_loss: 0.5043 | test_acc: 0.8117

Epoch: 48 | train_loss: 0.3935 | train_acc: 0.8510 | test_loss: 0.3686 | test_acc: 0.8505

Epoch: 49 | train_loss: 0.3467 | train_acc: 0.8591 | test_loss: 0.3680 | test_acc: 0.8511

Epoch: 50 | train_loss: 0.3517 | train_acc: 0.8631 | test_loss: 0.3234 | test_acc: 0.8830

In [ ]:
optimizer_1100 = torch.optim.Adam(params = model_1100.parameters(),
                             lr = 0.0005)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

train(model = model_1100,
      train_dataloader = train_dataloader_2_96,
      test_dataloader = test_dataloader_2_96,
      optimizer = optimizer_1100,
      loss_fn = loss_fn,
      best_acc = 0.9120,
      epochs = 30)

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 1.1345 | train_acc: 0.3451 | test_loss: 1.1167 | test_acc: 0.3742
Epoch: 2 | train_loss: 1.1480 | train_acc: 0.3475 | test_loss: 1.1144 | test_acc: 0.3754


KeyboardInterrupt: 

## Model 1200:  data augmented(25bins) , BatchNorm2d , 96 pixel images, New model structure

#### Maybe the data augmentation is too much?

In [ ]:
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader

augmented_transform_3_96 = transforms.Compose([
    transforms.Resize(size = (96, 96)),
    transforms.TrivialAugmentWide(num_magnitude_bins = 25),
    transforms.ToTensor()
])

simple_transform_96 = transforms.Compose([
    transforms.Resize(size = (96, 96)),
    transforms.ToTensor()
])

train_data_3_96 = datasets.ImageFolder(
    root = train_dir,
    transform = augmented_transform_3_96,
)

test_data_3_96 = datasets.ImageFolder(
    root = test_dir,
    transform = simple_transform_96
)

BATCH_SIZE = 16

train_dataloader_3_96 = DataLoader(
    dataset = train_data_2_96,
    batch_size = BATCH_SIZE,
    shuffle = True,
    num_workers = os.cpu_count()
)

test_dataloader_3_96 = DataLoader(
    dataset = test_data_3_96,
    batch_size = BATCH_SIZE,
    shuffle = False,
    num_workers = os.cpu_count()
)

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model_1200 = TinyVGG_2_96(input_shape = 3,
                          hidden_units = 64,
                          output_shape = 3).to(device)

In [ ]:
optimizer_1200 = torch.optim.Adam(params = model_1200.parameters(),
                             lr = 0.001)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

train(model = model_1200,
      train_dataloader = train_dataloader_3_96,
      test_dataloader = test_dataloader_3_96,
      optimizer = optimizer_1200,
      loss_fn = loss_fn,
      best_acc = 0.9120,
      epochs = 50)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 2.2181 | train_acc: 0.4111 | test_loss: 1.1511 | test_acc: 0.5589
Epoch: 2 | train_loss: 1.2545 | train_acc: 0.4926 | test_loss: 0.7953 | test_acc: 0.6524
Epoch: 3 | train_loss: 1.0081 | train_acc: 0.5463 | test_loss: 0.8980 | test_acc: 0.5612
Epoch: 4 | train_loss: 0.9455 | train_acc: 0.5656 | test_loss: 0.7087 | test_acc: 0.6955
Epoch: 5 | train_loss: 0.9726 | train_acc: 0.5674 | test_loss: 0.6811 | test_acc: 0.7257
Epoch: 6 | train_loss: 0.8974 | train_acc: 0.6035 | test_loss: 0.9789 | test_acc: 0.5691
Epoch: 7 | train_loss: 0.8535 | train_acc: 0.6277 | test_loss: 0.8248 | test_acc: 0.6208
Epoch: 8 | train_loss: 0.8338 | train_acc: 0.6300 | test_loss: 0.6456 | test_acc: 0.7542
Epoch: 9 | train_loss: 0.8233 | train_acc: 0.6430 | test_loss: 0.6605 | test_acc: 0.7162
Epoch: 10 | train_loss: 0.7776 | train_acc: 0.6697 | test_loss: 0.6803 | test_acc: 0.7092
Epoch: 11 | train_loss: 0.7962 | train_acc: 0.6576 | test_loss: 0.6050 | test_acc: 0.7604
Epoch: 12 | train_l

{'train_loss': [2.2180992889066116,
  1.2545372729605817,
  1.0081262537773619,
  0.9454552429787656,
  0.972588117663742,
  0.8974255657365137,
  0.8534631166897767,
  0.8337824270657613,
  0.823277289351673,
  0.7775920160273289,
  0.7962099687850221,
  0.7762845395304633,
  0.751727652676562,
  0.729685736886153,
  0.7342003332384934,
  0.7197442420408235,
  0.6853722701681421,
  0.7103363431937305,
  0.6593749089866665,
  0.6650679631436125,
  0.6622927741590121,
  0.6344438586251956,
  0.6127831402188497,
  0.6157431856114813,
  0.600169782414504,
  0.6089355374904389,
  0.574582207604503,
  0.5648119108262637,
  0.5449430857989805,
  0.5460458110410271,
  0.5469192300282472,
  0.540195827365767,
  0.5040285243844309,
  0.48105322088755614,
  0.5044486272208234,
  0.5044724887355845,
  0.456471804714372,
  0.4601981447520831,
  0.4707688507987252,
  0.4727009410131062,
  0.4702023817292342,
  0.42875526808466474,
  0.433636115692186,
  0.4061780812628303,
  0.4138214961222723,
  0

Epoch: 45 | train_loss: 0.4138 | train_acc: 0.8431 | test_loss: 0.4031 | test_acc: 0.8431

Epoch: 46 | train_loss: 0.4151 | train_acc: 0.8375 | test_loss: 0.3353 | test_acc: 0.8830

Epoch: 47 | train_loss: 0.3870 | train_acc: 0.8564 | test_loss: 0.3591 | test_acc: 0.8628

Epoch: 48 | train_loss: 0.4222 | train_acc: 0.8443 | test_loss: 0.3080 | test_acc: 0.8856

Epoch: 49 | train_loss: 0.3756 | train_acc: 0.8482 | test_loss: 0.7037 | test_acc: 0.7327

Epoch: 50 | train_loss: 0.3821 | train_acc: 0.8443 | test_loss: 0.3012 | test_acc: 0.8843

In [ ]:
optimizer_1200 = torch.optim.Adam(params = model_1200.parameters(),
                             lr = 0.0005)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

train(model = model_1200,
      train_dataloader = train_dataloader_3_96,
      test_dataloader = test_dataloader_3_96,
      optimizer = optimizer_1200,
      loss_fn = loss_fn,
      best_acc = 0.9120,
      epochs = 50)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 0.3425 | train_acc: 0.8645 | test_loss: 0.3161 | test_acc: 0.8735
Epoch: 2 | train_loss: 0.3101 | train_acc: 0.8827 | test_loss: 0.2952 | test_acc: 0.8843
Epoch: 3 | train_loss: 0.3189 | train_acc: 0.8847 | test_loss: 0.2903 | test_acc: 0.8843
Epoch: 4 | train_loss: 0.3125 | train_acc: 0.8836 | test_loss: 0.3018 | test_acc: 0.8763
Epoch: 5 | train_loss: 0.3120 | train_acc: 0.8869 | test_loss: 0.3470 | test_acc: 0.8706
Epoch: 6 | train_loss: 0.3114 | train_acc: 0.8865 | test_loss: 0.3238 | test_acc: 0.8830
Epoch: 7 | train_loss: 0.2999 | train_acc: 0.8882 | test_loss: 0.2868 | test_acc: 0.8963
Epoch: 8 | train_loss: 0.2913 | train_acc: 0.8874 | test_loss: 0.3299 | test_acc: 0.8720
Epoch: 9 | train_loss: 0.2706 | train_acc: 0.8951 | test_loss: 0.3396 | test_acc: 0.8775
Epoch: 10 | train_loss: 0.2824 | train_acc: 0.8991 | test_loss: 0.2806 | test_acc: 0.8976
Epoch: 11 | train_loss: 0.3050 | train_acc: 0.8835 | test_loss: 0.3093 | test_acc: 0.8816
Epoch: 12 | train_l

{'train_loss': [0.34251783354908016,
  0.3100764737431462,
  0.3189080823873374,
  0.3125110797267011,
  0.3120359621466474,
  0.31141344932112713,
  0.2999151978932374,
  0.291318435406854,
  0.27064951382418895,
  0.28235457203489667,
  0.3049711286757432,
  0.2550432524835387,
  0.26037919959912065,
  0.2535296620417994,
  0.25102182735312495,
  0.2671498594316819,
  0.2483565237386324,
  0.251995689678171,
  0.2560233599803549,
  0.23959106674536745,
  0.23742742819302048,
  0.2302874054971105,
  0.22840985922631643,
  0.24118865199439915,
  0.25326524294437247,
  0.22844355585093193,
  0.2282628422431278,
  0.22562160144778007,
  0.20092102075457996,
  0.2197231749297245,
  0.20895717937684227,
  0.2195452713400971,
  0.1975207859848408,
  0.1876404398081309,
  0.19825480993282288,
  0.21188728982939364,
  0.18285076550029694,
  0.19393754103187974,
  0.21004213898026564,
  0.2133906849375959,
  0.20911863302243938,
  0.20186993408393353,
  0.2012717117958352,
  0.1958211819700738

Epoch: 45 | train_loss: 0.1940 | train_acc: 0.9275 | test_loss: 0.3178 | test_acc: 0.8790

Epoch: 46 | train_loss: 0.1827 | train_acc: 0.9300 | test_loss: 0.3124 | test_acc: 0.8923

Epoch: 47 | train_loss: 0.1688 | train_acc: 0.9419 | test_loss: 0.3324 | test_acc: 0.8881

Epoch: 48 | train_loss: 0.2027 | train_acc: 0.9325 | test_loss: 0.3241 | test_acc: 0.8816

Epoch: 49 | train_loss: 0.1822 | train_acc: 0.9331 | test_loss: 0.3656 | test_acc: 0.8697

Epoch: 50 | train_loss: 0.1780 | train_acc: 0.9301 | test_loss: 0.3065 | test_acc: 0.8963

# Saving our final best performing model

In [2]:
from pathlib import Path

#1. Create model's directory
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents = True, exist_ok = True)

# 2. Create model save path
MODEL_NAME = "pizza_sushi_steak_best_model.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME


# 3. Save
print(f"Saving model to : {MODEL_SAVE_PATH}")
torch.save(obj=model_1100.state_dict(),
           f = MODEL_SAVE_PATH)

# I ended up implementing this in my model training function as it is a lot more
# convenient

# Conclusion

In the end, the best performing model ended up being model 1100 and I went from a 75% initial accuracy to a test accuracy of 91%.

I could have kept trying to improve the model, but it is quite time consuming and to make it better, I probably would have had to increase computation making it take even more time.

So, being content with my final model result I decided to end my experiment here.